# GNN-MARL Load-Shedding Framework -- Kaggle Single Notebook

End-to-end pipeline (config → data → forecaster → MARL → env → baselines → tuning → ablation → sensitivity → 2025-2040 projection → daily per-region report).

**Dataset:** `version_5.xlsx` -- real daily Bangladesh National Grid data (2019-11-21 to 2024-12-30), 9 regions.
Attach it to this notebook with **Add Input → Datasets**; Section 1 finds it automatically anywhere under `/kaggle/input/`. All outputs are written to `/kaggle/working/outputs/`.

**Run profile:** set `RUN_PROFILE` in Section 2 (`"smoke"` ≈ wiring check, `"fast"` ≈ default, `"full"` ≈ paper-grade budget). Turn on the GPU accelerator for the forecaster.

### Revision notes (what changed vs. the Colab version)
* **Paths:** Google Drive mount removed; dataset auto-discovered under `/kaggle/input`, outputs to `/kaggle/working/outputs`.
* **Data bugs fixed:** the raw file has 11 calendar gaps (2-7 days) -- rows are now re-indexed to a continuous daily calendar so `shift(7)` really means 7 days; windows whose targets fall on imputed days are excluded.
* **Region-order bug fixed:** windows were built in *alphabetical* region order while the adjacency matrix used `REGIONS` order, so the GCN was passing messages along the wrong edges. Everything is now pivoted explicitly in `REGIONS` order.
* **Window bugs fixed:** off-by-one in the window range, hard-coded `CFG.data.lookback_days` inside `make_windows`, and val/test windows that could not use the preceding history. Windows are now split by *target* dates, so no target ever crosses a split boundary.
* **Scaling added:** features and targets are standardised per region (train statistics only). The old unscaled MSE was dominated by Dhaka and trained very slowly (RMSE ≈ 1,800 MW).
* **Forecaster:** equivalent but ~N×D cheaper message passing, residual-to-last-observation head, early stopping, GPU support, and split-conformal calibration of the MC-Dropout intervals.
* **MAPPO bugs fixed:** log-probabilities were computed on the *unclamped* action but the *clamped* action was re-scored during the update (biased PPO ratio); advantages were just normalised rewards (no critic baseline, no GAE, episode boundaries ignored). Now: squashed-Gaussian policy with exact ratio, GAE(λ) from the centralised critic, minibatches, gradient clipping, best-checkpoint selection on validation.
* **Environment bugs fixed:** agents were scaling the *actual* (unknown) demand instead of the forecast; the supply constraint was a daily energy total (allowing energy to move between hours); fairness was computed on absolute MWh (penalising big regions); the cost term penalised *serving* load. Now: hourly capacity limit, served-ratio Jain fairness, over-dispatch cost, scale-free rewards.
* **Placeholders replaced with real runs:** the MAPPO tuning env (random numbers), the ablation stub (random numbers), and the report policy (MILP instead of MARL) now all use the real data and the trained models.
* **Baselines actually evaluated:** forecasting baselines (persistence, seasonal-naive, ridge, XGBoost) and scheduling baselines (historical realised, historical share, proportional, priority, greedy, LP oracle) are run on the 2024 test set with bootstrap CIs and paired tests.
* **Seasonal-naive bug fixed:** it used an hourly season (168) on daily data and returned an empty array for `season == horizon`.
* **Sensitivity fixes:** a *uniform* ±20% priority scaling leaves every ranking unchanged (the old sweep could not detect anything); replaced with independent per-region Monte-Carlo perturbations. The 3⁵ reward grid is replaced by a one-at-a-time sweep that is actually affordable. ENS reduction for the economics is measured, not assumed.
* **Projection/report:** seasonality and weekend effects calibrated from the data (the weekend discount was applied twice), capacity grows under an explicit supply scenario instead of staying flat for 16 years, fully vectorised (the old per-date filter was O(days²)), hourly schedules and reason codes written per region-day.

## 1. Setup: packages and Kaggle dataset paths

In [23]:
# Kaggle images already ship torch, xgboost, scikit-learn, scipy, pandas and
# openpyxl. Only install what is missing -- this also works with Internet OFF
# (the install is simply skipped if the package is already present).
import importlib, subprocess, sys

for _pkg in ["optuna", "xgboost", "openpyxl", "scipy", "sklearn"]:
    try:
        importlib.import_module(_pkg)
    except ImportError:
        _pip_name = "scikit-learn" if _pkg == "sklearn" else _pkg
        print(f"installing {_pip_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pip_name], check=False)
print("package check done")

package check done


In [24]:
"""
Kaggle paths
------------
1. Attach the dataset containing version_5.xlsx (Add Input -> Datasets).
   Kaggle mounts it read-only, e.g.
       /kaggle/input/<dataset-slug>/version_5.xlsx
   (newer layouts: /kaggle/input/datasets/<owner>/<dataset-slug>/version_5.xlsx)
   The file is found automatically; set RAW_XLSX_OVERRIDE only if you have
   several matching files attached.
2. Everything the notebook writes goes to /kaggle/working/outputs, which is
   kept when you "Save Version".
"""
import os, glob, json, random, time, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

KAGGLE_INPUT_DIR   = "/kaggle/input"
KAGGLE_WORKING_DIR = "/kaggle/working"
IS_KAGGLE = os.path.isdir(KAGGLE_INPUT_DIR)

RAW_XLSX_OVERRIDE = None     # e.g. "/kaggle/input/bd-grid-v5/version_5.xlsx"
DATASET_FILENAMES = ["version_5.xlsx", "version 5.xlsx", "version-5.xlsx"]


def locate_dataset() -> str:
    """Return the path of version_5.xlsx (explicit override > env var > search)."""
    explicit = RAW_XLSX_OVERRIDE or os.environ.get("LOADSHED_XLSX")
    if explicit:
        if not os.path.exists(explicit):
            raise FileNotFoundError(f"RAW_XLSX_OVERRIDE does not exist: {explicit}")
        return explicit
    roots = ([KAGGLE_INPUT_DIR] if IS_KAGGLE else []) + [os.getcwd()]
    for root in roots:
        for name in DATASET_FILENAMES:
            hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True))
            if hits:
                return hits[0]
        fuzzy = sorted(glob.glob(os.path.join(root, "**", "*version*5*.xls*"),
                                 recursive=True))
        if fuzzy:
            return fuzzy[0]
    raise FileNotFoundError(
        "Could not find version_5.xlsx.\n"
        "On Kaggle: click 'Add Input' -> Datasets and attach the dataset that "
        "contains version_5.xlsx (it will appear under /kaggle/input/...).\n"
        "Or set RAW_XLSX_OVERRIDE to the full path.")


PROJECT_DIR = KAGGLE_WORKING_DIR if IS_KAGGLE else os.getcwd()
OUTPUT_DIR  = f"{PROJECT_DIR}/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/per_region", exist_ok=True)

RAW_XLSX = locate_dataset()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed: int = 42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(42)

print("Running on Kaggle:", IS_KAGGLE)
print("Dataset         :", RAW_XLSX)
print("Output dir      :", OUTPUT_DIR)
print("Torch device    :", DEVICE,
      f"({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else "")

xls = pd.ExcelFile(RAW_XLSX)
print("\nAvailable sheets:")
for i, sheet in enumerate(xls.sheet_names):
    print(f"  {i}: {sheet}")

df = pd.read_excel(RAW_XLSX, sheet_name=0)
print("\nDataset loaded successfully")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

Running on Kaggle: True
Dataset         : /kaggle/input/datasets/mdlimonbinhossain/electricity-dataset/version 5.xlsx
Output dir      : /kaggle/working/outputs
Torch device    : cuda (Tesla T4)

Available sheets:
  0: Sheet1

Dataset loaded successfully
Shape: (1850, 45)

Columns:
['Date', 'Day of the week', 'Year', 'Month', 'Max. Demand at eve. peak (Generation end)', 'Max. Demand at eve. peak (Sub-station end)', 'Highest Generation (Generation end)', 'Minimum Generation (Generation end)', 'Day-peak Generation (Generation end)', 'Evening-peak Generation (Generation end)', 'Minimum Generation Forecast up to 8:00 hrs.', 'Maximum Temperature in Dhaka was', 'Gas/LF limitation', 'Coal supply Limitation', 'Low water level in Kaptai lake', 'Plants under shut down/ maintenance', 'Dhaka_demand', 'Dhaka_supply', 'Dhaka_load', 'Chattogram_demand', 'Chattogram_supply', 'Chattogram_load', 'Rajshahi_demand', 'Rajshahi_supply', 'Rajshahi_load', 'Mymensingh_demand', 'Mymensingh_supply', 'Mymensingh_l

,Date,Day of the week,Year,Month,Max. Demand at eve. peak (Generation end),Max. Demand at eve. peak (Sub-station end),Highest Generation (Generation end),Minimum Generation (Generation end),Day-peak Generation (Generation end),Evening-peak Generation (Generation end),Minimum Generation Forecast up to 8:00 hrs.,Maximum Temperature in Dhaka was,Gas/LF limitation,Coal supply Limitation,Low water level in Kaptai lake,Plants under shut down/ maintenance,Dhaka_demand,Dhaka_supply,Dhaka_load,Chattogram_demand,Chattogram_supply,Chattogram_load,Rajshahi_demand,Rajshahi_supply,Rajshahi_load,Mymensingh_demand,Mymensingh_supply,Mymensingh_load,Sylhet_demand,Sylhet_supply,Sylhet_load,Barishal_demand,Barishal_supply,Barishal_load,Rangpur_demand,Rangpur_supply,Rangpur_load,Cumilla_demand,Cumilla_supply,Cumilla_load,Khulna_demand,Khulna_supply,Khulna_load,Holiday name,Holiday_cat
0,2019-11-21,Thursday,2019,11,8785,8110.0,8785.0,5647.0,7294.7,8785,5475.9,29.6,1349,0,0,470.0,3214,3214,0,905,905,0,953,953,0,639,639,0,329,329,0,208,208,0,208,208,0,701,701,0,953,953,0,No Holiday,0
1,2019-11-22,Friday,2019,11,7886,7344.0,7886.0,5283.5,5725.9,7886,5200.0,29.3,1164,0,0,470.0,2686,2686,0,813,813,0,908,908,0,613,613,0,320,320,0,201,201,0,201,201,0,694,694,0,908,908,0,No Holiday,0
2,2019-11-23,Saturday,2019,11,8827,8197.0,8827.0,5200.0,6731.7,8946,5470.0,28.8,1114,0,0,470.0,3223,3223,0,897,897,0,974,974,0,652,652,0,328,328,0,210,210,0,210,210,0,729,729,0,974,974,0,No Holiday,0
3,2019-11-24,Sunday,2019,11,8855,8224.0,8855.0,5470.0,7062.8,8855,5500.0,29.1,1204,0,0,555.0,3248,3248,0,883,883,0,983,983,0,646,646,0,327,327,0,216,216,0,216,216,0,722,722,0,983,983,0,No Holiday,0
4,2019-11-25,Monday,2019,11,8982,8356.0,8978.0,5500.0,7110.7,8982,5609.0,29.0,1255,0,0,555.0,3346,3346,0,878,878,0,981,981,0,669,669,0,328,328,0,219,219,0,219,219,0,735,735,0,981,981,0,No Holiday,0


## 2. Configuration (all knobs surfaced here)

Region names and raw-column mappings below match `version_5.xlsx` exactly (e.g. `Cumilla_demand`, `Gas/LF limitation`). `RUN_PROFILE` controls the compute budget of every tuning / training step.

In [25]:
"""
config.py
=========
Centralized configuration for the GNN-MARL load-shedding framework.

Design principle
----------------
All numerical constants that a reviewer or user might legitimately question
(reward weights, discount rate, VOLL, priority scores, seeds, budgets, grid
assumptions) live here so that sensitivity analysis is a one-line change and
nothing is hard-coded deep inside a module.

The reviewer explicitly asked for:
  - reward-weight sensitivity           (Comment 8)
  - priority-score sensitivity          (Comment 19)
  - economic sensitivity                (Comment 15)
This file exposes every one of those knobs.
"""

from dataclasses import dataclass, field
from typing import Dict, List, Tuple


# --------------------------------------------------------------------------- #
# 0. Compute budget -- "smoke" (wiring check), "fast" (default), "full"
# --------------------------------------------------------------------------- #
RUN_PROFILE: str = os.environ.get("LOADSHED_PROFILE", "fast")


# --------------------------------------------------------------------------- #
# 1. Regions and graph structure
# --------------------------------------------------------------------------- #
# NOTE: spelling matches the real dataset (version_5.xlsx) column prefixes
# exactly, e.g. "Cumilla_demand", "Cumilla_supply", "Cumilla_load".
# This ORDER is the node order of the graph -- every array in the pipeline is
# pivoted into exactly this order.
REGIONS: List[str] = [
    "Dhaka", "Chattogram", "Khulna", "Rajshahi", "Sylhet",
    "Barishal", "Rangpur", "Mymensingh", "Cumilla",
]
N_REGIONS: int = len(REGIONS)


# --------------------------------------------------------------------------- #
# 1b. Raw dataset column map (version_5.xlsx, sheet "Sheet1", 1850 daily rows,
#     2019-11-21 -> 2024-12-30).
# --------------------------------------------------------------------------- #
RAW_DATE_COL: str = "Date"

# National-level columns that are KNOWN before the day starts.
RAW_NATIONAL_SAFE: Dict[str, str] = {
    "Day of the week": "day_of_week_name",
    "Year": "year",
    "Month": "month",
    "Maximum Temperature in Dhaka was": "temperature",
    "Holiday name": "holiday_name",
    "Holiday_cat": "holiday_cat",
}
# Same-day operational OUTCOMES -- NOT known when a day-ahead forecast is
# issued, so excluded from the forecaster (Reviewer Comment 6).
RAW_NATIONAL_REALIZED_LEAKY: Dict[str, str] = {
    "Gas/LF limitation": "gas_limitation",
    "Coal supply Limitation": "coal_limitation",
    "Low water level in Kaptai lake": "water_level_low",
    "Plants under shut down/ maintenance": "plant_shutdown",
    "Max. Demand at eve. peak (Generation end)": "peak_demand_gen_end",
    "Max. Demand at eve. peak (Sub-station end)": "peak_demand_substation_end",
    "Highest Generation (Generation end)": "highest_generation",
    "Minimum Generation (Generation end)": "minimum_generation",
    "Day-peak Generation (Generation end)": "day_peak_generation",
    "Evening-peak Generation (Generation end)": "evening_peak_generation",
    "Minimum Generation Forecast up to 8:00 hrs.": "min_generation_forecast_8am",
}

# Per-region columns "{Region}_demand/_supply/_load" (MW at the evening peak,
# sub-station end; demand - supply = load within +-2 MW in the raw file).
# "_demand" is the forecasting target. "_supply"/"_load" are same-day REALIZED
# outcomes: excluded from the forecaster, but kept separately as the REAL
# historical baseline for the scheduling comparison (Reviewer Comment 12).
REGION_COL_SUFFIXES: Dict[str, str] = {
    "demand": "demand_mw",
    "supply": "supply_mw_realized",
    "load":   "load_shed_mw_realized",
}


# Physical adjacency of the Bangladesh National Grid (Reviewer Comment 3).
GRID_EDGES: List[Tuple[str, str, Dict[str, float]]] = [
    ("Dhaka", "Chattogram",  {"capacity_mw": 1200, "kv": 400}),
    ("Dhaka", "Cumilla",     {"capacity_mw":  900, "kv": 230}),
    ("Dhaka", "Mymensingh",  {"capacity_mw":  700, "kv": 230}),
    ("Dhaka", "Rajshahi",    {"capacity_mw":  800, "kv": 230}),
    ("Dhaka", "Khulna",      {"capacity_mw":  900, "kv": 230}),
    ("Dhaka", "Sylhet",      {"capacity_mw":  600, "kv": 230}),
    ("Chattogram", "Cumilla", {"capacity_mw": 700, "kv": 230}),
    ("Cumilla", "Sylhet",    {"capacity_mw":  500, "kv": 132}),
    ("Rajshahi", "Rangpur",  {"capacity_mw":  600, "kv": 230}),
    ("Rajshahi", "Khulna",   {"capacity_mw":  500, "kv": 230}),
    ("Khulna", "Barishal",   {"capacity_mw":  400, "kv": 132}),
    ("Mymensingh", "Sylhet", {"capacity_mw":  400, "kv": 132}),
    ("Mymensingh", "Rangpur",{"capacity_mw":  500, "kv": 230}),
    ("Barishal", "Dhaka",    {"capacity_mw":  400, "kv": 132}),
]


# --------------------------------------------------------------------------- #
# 2. Data / leakage-safety configuration  (Reviewer Comment 6)
# --------------------------------------------------------------------------- #
@dataclass
class DataConfig:
    """
    version_5.xlsx is DAILY data. The forecast for day t+1..t+7 is issued at
    the end of day t, so day t's own demand ("demand_obs") is known and safe.
    """
    observed_at_decision_time: List[str] = field(default_factory=lambda: [
        "day_of_week_num", "month", "year", "is_weekend",
        "doy_sin", "doy_cos",       # smooth annual seasonality (calendar)
        "temperature_forecast",     # day-ahead weather forecast proxy
        "holiday_cat",              # calendar holidays are known in advance
        "demand_obs",               # demand of the forecast-issue day t
        "demand_lag_1d", "demand_lag_7d", "demand_lag_30d",
        "rolling_mean_7d", "rolling_std_7d",
    ])
    forbidden_leaky: List[str] = field(default_factory=lambda: [
        "gas_limitation", "coal_limitation", "water_level_low",
        "plant_shutdown", "peak_demand_gen_end", "peak_demand_substation_end",
        "highest_generation", "minimum_generation", "day_peak_generation",
        "evening_peak_generation", "min_generation_forecast_8am",
        "supply_mw_realized", "load_shed_mw_realized",
    ])
    lookback_days: int = 60
    horizon_days:  int = 7
    max_interp_gap_days: int = 7            # longest calendar gap in the file
    train_end_date: str = "2022-12-31"
    val_end_date:   str = "2023-12-31"
    test_end_date:  str = "2024-12-30"
    projection_start: str = "2025-01-01"
    projection_end:   str = "2040-12-31"


# --------------------------------------------------------------------------- #
# 3. Reward weights  (Reviewer Comment 8 asks for sensitivity)
#    All reward terms are scale-free (percent of the day's demand) so the
#    weights are comparable across years and regions.
# --------------------------------------------------------------------------- #
@dataclass
class RewardWeights:
    alpha_ens:      float = 1.00   # penalize energy-not-served (% of demand)
    beta_priority:  float = 0.60   # penalize priority-weighted shedding (%)
    gamma_fair:     float = 0.40   # penalize (1 - Jain fairness of served ratio)
    delta_cost:     float = 0.20   # penalize over-dispatch of SCARCE capacity (binding hours)
    epsilon_smooth: float = 0.10   # penalize day-to-day switching of the curve

    _SHORT = {"alpha": "alpha_ens", "beta": "beta_priority",
              "gamma": "gamma_fair", "delta": "delta_cost",
              "epsilon": "epsilon_smooth"}

    def as_dict(self) -> Dict[str, float]:
        return {k: getattr(self, v) for k, v in self._SHORT.items()}

    @classmethod
    def from_short(cls, d: Dict[str, float]) -> "RewardWeights":
        return cls(**{cls._SHORT[k]: float(v) for k, v in d.items()})


# --------------------------------------------------------------------------- #
# 4. Regional priority scores  (Reviewer Comment 19)
# --------------------------------------------------------------------------- #
# EXPERT-DEFINED tiers based on load composition (critical loads, hospitals,
# industrial share). Not learned. The sensitivity module perturbs them.
PRIORITY_SCORES_EXPERT: Dict[str, float] = {
    "Dhaka":      0.95,
    "Chattogram": 0.90,
    "Sylhet":     0.75,
    "Khulna":     0.70,
    "Rajshahi":   0.65,
    "Cumilla":    0.60,
    "Mymensingh": 0.55,
    "Rangpur":    0.50,
    "Barishal":   0.45,
}


# --------------------------------------------------------------------------- #
# 5. MARL (MAPPO) hyperparameters -- defaults; overridden by Optuna
# --------------------------------------------------------------------------- #
@dataclass
class MARLConfig:
    n_agents: int = N_REGIONS
    obs_dim_per_agent: int = 32          # set at runtime = GCN dim + extras
    action_dim_per_agent: int = 24       # 24 hourly dispatch fractions
    action_max: float = 1.3              # may request up to 130% of forecast
    action_init: float = 1.10            # initial request (hedge vs forecast error)
    # The 24-hour curve is generated from n_action_basis smooth radial-basis
    # bumps (learned weights, no hand-picked hours). 24 independent Gaussian
    # dimensions per agent (216 in total) gave too little signal per
    # dimension for PPO; 0 = use the raw 24-dim curve (ablation).
    n_action_basis: int = 6
    init_log_std: float = -1.5
    obs_embed_dim: int = 0               # PCA dims of the GCN embedding observed; tuned {0,2,4}
    hidden_dim: int = 128
    lr_actor: float = 3e-4
    lr_critic: float = 1e-3
    # A day's dispatch curve only affects that day (plus a small smoothing
    # term), so the default is a contextual-bandit discount. With gamma=0.99
    # the advantage was dominated by ~17 future days of exogenous conditions.
    gamma: float = 0.0
    gae_lambda: float = 0.0
    clip_ratio: float = 0.2
    entropy_coef: float = 0.001
    max_grad_norm: float = 0.5
    n_epochs_per_update: int = 8
    rollout_len: int = 256               # env steps (days) per PPO update
    minibatch_size: int = 64
    episode_len: int = 64                # days per training episode
    # Credit assignment: each agent optimises
    #   r_i = w * team_reward + (1 - w) * local_reward_i
    # with a per-agent value head on the centralised critic (w=1 -> pure
    # shared-reward MAPPO, available for ablation).
    team_reward_weight: float = 1.0
    # Episode starts are drawn with weight (1 + stress_start_weight * #stress
    # days in the episode) -- only 11% of 2020-2022 training days are short.
    stress_start_weight: float = 1.0
    total_env_steps: int = 500_000
    seeds: List[int] = field(default_factory=lambda: [7, 13, 29, 42, 101])


# --------------------------------------------------------------------------- #
# 6. Grid-operation assumptions used by the environment and the projection
# --------------------------------------------------------------------------- #
@dataclass
class GridOpsConfig:
    # On days WITHOUT recorded shedding the true available capacity is above
    # served demand; the dataset only shows served MW, so a reserve margin is
    # assumed. On stress days capacity = realised supply (exactly observed).
    unconstrained_reserve_margin: float = 0.05
    # Hourly profile (normalised so the evening-peak hour = 1.0, because the
    # regional columns are evening-peak MW).
    profile_base: float = 0.62
    profile_day_amp: float = 0.18
    profile_evening_amp: float = 0.22
    profile_peak_hour: int = 19
    # Projection supply plan: capacity = P95 of 2024 served peak, growing at a
    # planned capacity-expansion CAGR (scenario assumption, NOT data).
    supply_base_quantile: float = 0.95
    supply_cagr: float = 0.060
    stress_threshold_mw: float = 0.5


# --------------------------------------------------------------------------- #
# 7. Economic assumptions  (Reviewer Comment 15 asks for sensitivity)
# --------------------------------------------------------------------------- #
@dataclass
class EconomicConfig:
    voll_bdt_per_kwh: float = 45.0          # value of lost load
    implementation_cost_bdt: float = 0.5e9  # ASSUMED; sensitivity 0.25e9 .. 2.0e9
    annual_opex_bdt: float = 0.05e9
    discount_rate: float = 0.10
    horizon_years: int = 10


# --------------------------------------------------------------------------- #
# 8. Compute budget
# --------------------------------------------------------------------------- #
@dataclass
class RunBudget:
    profile: str = "fast"
    fc_trials: int = 12
    fc_tune_epochs: int = 25
    fc_final_epochs: int = 80
    fc_patience: int = 10
    mc_samples: int = 50
    marl_trials: int = 8
    marl_tune_seeds: int = 2
    marl_tune_steps: int = 15_000
    marl_final_steps: int = 60_000
    ablation_seeds: List[int] = field(default_factory=lambda: [7, 13, 29])
    ablation_fc_epochs: int = 40
    ablation_marl_steps: int = 20_000
    run_reward_sweep: bool = True
    reward_sweep_steps: int = 10_000
    priority_mc_draws: int = 200


def make_budget(profile: str) -> RunBudget:
    if profile == "smoke":
        return RunBudget(profile="smoke", fc_trials=2, fc_tune_epochs=2,
                         fc_final_epochs=3, fc_patience=2, mc_samples=8,
                         marl_trials=2, marl_tune_seeds=1, marl_tune_steps=600,
                         marl_final_steps=1_200, ablation_seeds=[7],
                         ablation_fc_epochs=2, ablation_marl_steps=600,
                         run_reward_sweep=True, reward_sweep_steps=300,
                         priority_mc_draws=20)
    if profile == "full":
        return RunBudget(profile="full", fc_trials=40, fc_tune_epochs=40,
                         fc_final_epochs=150, fc_patience=15, mc_samples=100,
                         marl_trials=25, marl_tune_seeds=3,
                         marl_tune_steps=40_000, marl_final_steps=300_000,
                         ablation_seeds=[7, 13, 29, 42, 101],
                         ablation_fc_epochs=80, ablation_marl_steps=80_000,
                         run_reward_sweep=True, reward_sweep_steps=40_000,
                         priority_mc_draws=1000)
    return RunBudget(profile="fast")


# --------------------------------------------------------------------------- #
# 9. Global assembly
# --------------------------------------------------------------------------- #
@dataclass
class GlobalConfig:
    data:     DataConfig     = field(default_factory=DataConfig)
    reward:   RewardWeights  = field(default_factory=RewardWeights)
    marl:     MARLConfig     = field(default_factory=MARLConfig)
    grid:     GridOpsConfig  = field(default_factory=GridOpsConfig)
    economic: EconomicConfig = field(default_factory=EconomicConfig)
    budget:   RunBudget      = field(default_factory=lambda: make_budget(RUN_PROFILE))
    output_dir: str = "./outputs"


CFG = GlobalConfig()
PRIO_VEC = np.array([PRIORITY_SCORES_EXPERT[r] for r in REGIONS], dtype=np.float64)

In [26]:
# Point outputs at the Kaggle working directory set up in Section 1
CFG.output_dir = OUTPUT_DIR
print("CFG.output_dir set to:", CFG.output_dir)
print("Run profile          :", CFG.budget.profile)
print("Budget               :", CFG.budget)

CFG.output_dir set to: /kaggle/working/outputs
Run profile          : fast
Budget               : RunBudget(profile='fast', fc_trials=12, fc_tune_epochs=25, fc_final_epochs=80, fc_patience=10, mc_samples=50, marl_trials=8, marl_tune_seeds=2, marl_tune_steps=15000, marl_final_steps=60000, ablation_seeds=[7, 13, 29], ablation_fc_epochs=40, ablation_marl_steps=20000, run_reward_sweep=True, reward_sweep_steps=10000, priority_mc_draws=200)


## 3. Leakage-safe data loader

In [27]:
"""
data/loader.py
==============
Leakage-safe data loader for version_5.xlsx.

The real dataset is DAILY and WIDE: one row per calendar day with national
columns plus "{Region}_demand/_supply/_load". load_raw() regularises it to a
continuous daily calendar, build_leakage_safe_features() melts it into the
long (timestamp, region, ...) format and keeps only decision-time features,
and build_window_dataset() produces (lookback, horizon) windows in REGIONS
order, split chronologically by TARGET date.

Reviewer Comment 6: gas/coal limitation, plant shutdown, generation figures,
regional supply and load are same-day realised outcomes -> never features.

Public functions
----------------
load_raw(path)                         -> wide daily frame (continuous calendar)
build_leakage_safe_features(df)        -> long frame, only observed-at-t vars
extract_realized(df)                   -> long frame of realised supply / shed
train_val_test_split(df)               -> row-level chronological split
build_window_dataset(feats)            -> panel arrays + split window indices
windows_for(wd, split)                 -> (X, y) for one split
PanelScaler                            -> per-region standardisation
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple


def _regularize_daily(df: pd.DataFrame, max_gap: int) -> pd.DataFrame:
    """Re-index to a continuous daily calendar (the raw file has 11 gaps of
    2-7 days, so shift(k) would otherwise NOT mean k days) and interpolate
    numeric columns across the gaps. Imputed rows are flagged."""
    df = df.sort_values(RAW_DATE_COL).drop_duplicates(RAW_DATE_COL, keep="last")
    original = pd.DatetimeIndex(df[RAW_DATE_COL])
    full = pd.date_range(original.min(), original.max(), freq="D")
    df = df.set_index(RAW_DATE_COL).reindex(full)
    df.index.name = RAW_DATE_COL
    imputed = ~full.isin(original)

    skip = {"Year", "Month", "Holiday_cat", "is_imputed"}
    num_cols = [c for c in df.select_dtypes(include="number").columns
                if c not in skip]
    df[num_cols] = df[num_cols].interpolate(method="time", limit=max_gap,
                                            limit_area="inside")
    df["Day of the week"] = df.index.day_name()
    df["Year"] = df.index.year
    df["Month"] = df.index.month
    if "Holiday name" in df.columns:
        df["Holiday name"] = df["Holiday name"].fillna("No Holiday")
    if "Holiday_cat" in df.columns:
        df["Holiday_cat"] = df["Holiday_cat"].fillna(0).astype(int)
    prev = df["is_imputed"].fillna(0).astype(int) if "is_imputed" in df else 0
    df["is_imputed"] = np.maximum(prev, imputed.astype(int))
    return df.reset_index()


def load_raw(path: str, sheet_name=0, cfg: DataConfig = CFG.data) -> pd.DataFrame:
    """Reads version_5.xlsx (wide, one row per day) and regularises the calendar."""
    df = pd.read_excel(path, sheet_name=sheet_name)
    df.columns = [str(c).strip() for c in df.columns]
    df[RAW_DATE_COL] = pd.to_datetime(df[RAW_DATE_COL])
    n_before = len(df)
    df = _regularize_daily(df, cfg.max_interp_gap_days)
    print(f"load_raw: {n_before} raw rows -> {len(df)} calendar days "
          f"({int(df['is_imputed'].sum())} gap days interpolated)")
    return df


def build_leakage_safe_features(df: pd.DataFrame, cfg: DataConfig = CFG.data,
                                require_realized: bool = True) -> pd.DataFrame:
    """
    Melts the wide daily file into long (timestamp, region, ...) format with
    ONLY variables available when the day-ahead forecast is issued.
    Columns returned: timestamp, region, demand_mw, is_imputed + features.
    """
    df = df.copy()
    df = df.rename(columns=RAW_NATIONAL_SAFE)
    df = df.rename(columns=RAW_NATIONAL_REALIZED_LEAKY)       # dropped below
    ts = pd.to_datetime(df[RAW_DATE_COL])
    df["day_of_week_num"] = ts.dt.dayofweek                   # 0=Mon..6=Sun
    df["is_weekend"] = ts.dt.dayofweek.isin([4, 5]).astype(int)   # Fri+Sat (BD)
    doy = ts.dt.dayofyear.values
    df["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    df["year"] = ts.dt.year
    df["month"] = ts.dt.month
    if "holiday_cat" not in df.columns:
        df["holiday_cat"] = 0
    if "is_imputed" not in df.columns:
        df["is_imputed"] = 0

    national_cols = ([RAW_DATE_COL, "day_of_week_num", "is_weekend", "year",
                      "month", "doy_sin", "doy_cos", "temperature",
                      "holiday_cat", "is_imputed"]
                     + list(RAW_NATIONAL_REALIZED_LEAKY.values()))
    national_cols = [c for c in national_cols if c in df.columns]

    long_frames = []
    for region in REGIONS:
        dcol = f"{region}_demand"
        if dcol not in df.columns:
            raise KeyError(f"Column '{dcol}' not found. Check REGIONS spelling "
                           f"against the workbook headers.")
        cols = {f"{region}_{suf}": canon for suf, canon in REGION_COL_SUFFIXES.items()
                if f"{region}_{suf}" in df.columns}
        if require_realized and len(cols) < len(REGION_COL_SUFFIXES):
            raise KeyError(f"Missing supply/load columns for region '{region}'.")
        sub = df[[*national_cols, *cols.keys()]].rename(columns=cols)
        sub = sub.rename(columns={RAW_DATE_COL: "timestamp"})
        sub["region"] = region
        long_frames.append(sub)
    long_df = pd.concat(long_frames, ignore_index=True)
    long_df = long_df.sort_values(["region", "timestamp"]).reset_index(drop=True)
    g = long_df.groupby("region")

    # Day-ahead weather FORECAST proxy: next day's realised max temperature
    # (the file has no forecast column). At issue time t this is the forecast
    # for the first target day t+1.
    long_df["temperature_forecast"] = g["temperature"].shift(-1)

    # Demand of the issue day itself and its lags (all strictly <= t).
    long_df["demand_obs"] = long_df["demand_mw"]
    for lag_d, name in [(1, "demand_lag_1d"), (7, "demand_lag_7d"),
                        (30, "demand_lag_30d")]:
        long_df[name] = g["demand_mw"].shift(lag_d)
    long_df["rolling_mean_7d"] = g["demand_mw"].transform(
        lambda s: s.shift(1).rolling(7, min_periods=3).mean())
    long_df["rolling_std_7d"] = g["demand_mw"].transform(
        lambda s: s.shift(1).rolling(7, min_periods=3).std())

    # ---- Enforce leakage safety ----
    feats = list(cfg.observed_at_decision_time)
    leaked = set(feats) & set(cfg.forbidden_leaky)
    assert not leaked, f"Leaky features configured as safe: {leaked}"
    for banned in cfg.forbidden_leaky:
        if banned in long_df.columns:
            long_df = long_df.drop(columns=[banned])
    keep = ["timestamp", "region", "demand_mw", "is_imputed"] + feats
    long_df = long_df[keep].dropna(subset=feats + ["demand_mw"]).reset_index(drop=True)
    return long_df


def extract_realized(df: pd.DataFrame) -> pd.DataFrame:
    """Long frame of REALISED outcomes (supply served, load shed). Used ONLY as
    the historical scheduling baseline and the environment's capacity -- never
    as a forecaster input."""
    rows = []
    for region in REGIONS:
        sub = pd.DataFrame({
            "timestamp": pd.to_datetime(df[RAW_DATE_COL]),
            "region": region,
            "demand_mw": df[f"{region}_demand"].astype(float),
            "supply_mw_realized": df[f"{region}_supply"].astype(float),
            "load_shed_mw_realized": df[f"{region}_load"].astype(float).clip(lower=0),
            "is_imputed": df.get("is_imputed", 0),
        })
        rows.append(sub)
    return pd.concat(rows, ignore_index=True)


def train_val_test_split(df: pd.DataFrame, cfg: DataConfig = CFG.data
                         ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Row-level chronological split -- no random shuffling (avoids look-ahead)."""
    t = pd.to_datetime(df["timestamp"])
    train = df[t <= cfg.train_end_date]
    val   = df[(t > cfg.train_end_date) & (t <= cfg.val_end_date)]
    test  = df[(t > cfg.val_end_date) & (t <= cfg.test_end_date)]
    return train, val, test


def panel_arrays(feats: pd.DataFrame, feature_cols: List[str]):
    """Pivot the long frame into dense arrays in REGIONS order.
    Returns F (T, N, n_feat), Y (T, N), imputed (T,), dates (T,)."""
    def pv(col):
        return (feats.pivot(index="timestamp", columns="region", values=col)
                     .reindex(columns=REGIONS))
    y_pv = pv("demand_mw").dropna()
    dates = y_pv.index
    step = np.diff(dates.values).astype("timedelta64[D]").astype(int)
    if len(step) and (step != 1).any():
        raise ValueError("Feature frame is not a continuous daily calendar.")
    F = np.stack([pv(c).loc[dates].values for c in feature_cols], axis=-1)
    Y = y_pv.values
    imp = pv("is_imputed").loc[dates].max(axis=1).values.astype(int)
    return F.astype(np.float32), Y.astype(np.float32), imp, pd.DatetimeIndex(dates)


def build_window_dataset(feats: pd.DataFrame, cfg: DataConfig = CFG.data,
                         feature_cols: List[str] | None = None) -> Dict:
    """
    Every origin day t (forecast issued at end of day t) gives one sample:
        X = features of days t-L+1 .. t      (L, N, F)
        y = demand of days t+1 .. t+H        (N, H)
    A window belongs to a split only if ALL its target days are inside that
    split; the lookback may reach into earlier splits (that is past data, so
    no leakage). Windows whose targets include interpolated gap days are
    dropped from every split.
    """
    feature_cols = feature_cols or list(cfg.observed_at_decision_time)
    F, Y, imp, dates = panel_arrays(feats, feature_cols)
    L, H = cfg.lookback_days, cfg.horizon_days
    origins = np.arange(L - 1, len(dates) - H)
    t_first = dates[origins + 1]
    t_last = dates[origins + H]
    tgt_imp = np.array([imp[o + 1:o + H + 1].any() for o in origins])

    tr_end, va_end, te_end = (pd.Timestamp(cfg.train_end_date),
                              pd.Timestamp(cfg.val_end_date),
                              pd.Timestamp(cfg.test_end_date))
    ok = ~tgt_imp
    splits = {
        "train": origins[ok & (t_last <= tr_end)],
        "val":   origins[ok & (t_first > tr_end) & (t_last <= va_end)],
        "test":  origins[ok & (t_first > va_end) & (t_last <= te_end)],
    }
    return {"F": F, "Y": Y, "imputed": imp, "dates": dates,
            "feature_cols": feature_cols, "splits": splits,
            "lookback": L, "horizon": H}


def gather_windows(F: np.ndarray, origins: np.ndarray, lookback: int) -> np.ndarray:
    """(T, N, F) -> (S, L, N, F) windows ending at each origin."""
    idx = origins[:, None] - lookback + 1 + np.arange(lookback)[None, :]
    return F[idx]


def windows_for(wd: Dict, split: str, scaler: "PanelScaler | None" = None
                ) -> Tuple[np.ndarray, np.ndarray]:
    """X: (S, L, N, F) float32   y: (S, N, H) float32 (scaled if scaler given)."""
    o = wd["splits"][split]
    X = gather_windows(wd["F"], o, wd["lookback"])
    tidx = o[:, None] + 1 + np.arange(wd["horizon"])[None, :]
    y = wd["Y"][tidx].transpose(0, 2, 1)                       # (S, N, H)
    if scaler is not None:
        X, y = scaler.transform_x(X), scaler.transform_y(y)
    return X.astype(np.float32), y.astype(np.float32)


class PanelScaler:
    """Per-region, per-feature standardisation fitted on TRAIN days only.
    The demand_obs feature shares the target's statistics so the model can
    learn a residual on top of the last observation."""
    def fit(self, wd: Dict, train_end: str, residual_feature: str = "demand_obs"):
        mask = wd["dates"] <= pd.Timestamp(train_end)
        F, Y = wd["F"][mask], wd["Y"][mask]
        self.mu_x = F.mean(axis=0); self.sd_x = F.std(axis=0)
        self.sd_x[self.sd_x < 1e-6] = 1.0
        self.mu_y = Y.mean(axis=0); self.sd_y = Y.std(axis=0) + 1e-6
        self.residual_idx = None
        if residual_feature in wd["feature_cols"]:
            k = wd["feature_cols"].index(residual_feature)
            self.mu_x[:, k], self.sd_x[:, k] = self.mu_y, self.sd_y
            self.residual_idx = k
        return self

    def transform_x(self, X):   return (X - self.mu_x) / self.sd_x
    def transform_y(self, y):   return (y - self.mu_y[:, None]) / self.sd_y[:, None]
    def inverse_y(self, y):     return y * self.sd_y[:, None] + self.mu_y[:, None]
    def inverse_scale(self, s): return s * self.sd_y[:, None]      # widths

## 4. GNN-LSTM forecaster with MC-Dropout

In [28]:
"""
models/forecaster.py
====================
GNN-LSTM forecaster with MC-Dropout predictive intervals.

Addresses Reviewer Comments
---------------------------
Comment 2  : GCN parameters are LEARNED (not random) via supervised joint
             training with the LSTM head (freeze_gcn() gives the random-GCN
             ablation).
Comment 3  : GCN uses edge attributes (transmission capacity, kV class).
Comment 5  : MC-Dropout predictive intervals + PICP / MPIW / Winkler, plus a
             split-conformal recalibration on the validation set.
"""

from __future__ import annotations
import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple


# --------------------------------------------------------------------------- #
# Attributed GCN layer
# --------------------------------------------------------------------------- #
class AttrGCNLayer(nn.Module):
    """
    Message m_ij = W_s h_i + W_n h_j + b + E(e_ij), mean-aggregated over the
    neighbours j of i. This is algebraically identical to the previous
    Linear([h_i, h_j]) formulation but costs O(B*N*D^2 + B*N^2*D) instead of
    O(B*N^2*D^2), which is what made one Optuna trial take ~14 minutes.
    """
    def __init__(self, in_dim: int, out_dim: int, edge_attr_dim: int):
        super().__init__()
        self.node_proj = nn.Linear(in_dim, out_dim)
        self.edge_proj = nn.Linear(edge_attr_dim, out_dim)
        self.msg_self  = nn.Linear(out_dim, out_dim)             # W_s, b
        self.msg_nbr   = nn.Linear(out_dim, out_dim, bias=False)  # W_n

    def forward(self, h: torch.Tensor, adj: torch.Tensor,
                edge_attr: torch.Tensor) -> torch.Tensor:
        """
        h         : (B, N, in_dim)
        adj       : (N, N) 0/1 mask
        edge_attr : (N, N, edge_attr_dim) zero where no edge
        """
        hp = self.node_proj(h)                                   # (B, N, D)
        deg = adj.sum(dim=1, keepdim=True).clamp(min=1.0)        # (N, 1)
        self_term = self.msg_self(hp) * deg                      # (B, N, D)
        nbr_term = torch.einsum("ij,bjd->bid", adj, self.msg_nbr(hp))
        edge_term = (adj.unsqueeze(-1) * self.edge_proj(edge_attr)).sum(dim=1)
        return F.relu((self_term + nbr_term + edge_term) / deg)


# --------------------------------------------------------------------------- #
# Full model: GCN encoder + per-region LSTM decoder
# --------------------------------------------------------------------------- #
class GNNLSTMForecaster(nn.Module):
    def __init__(self, n_features: int, n_regions: int,
                 lookback: int, horizon: int,
                 edge_attr_dim: int = 2,
                 gcn_hidden: int = 64, lstm_hidden: int = 128,
                 dropout: float = 0.2, residual_idx: int | None = None):
        super().__init__()
        self.n_regions = n_regions
        self.horizon = horizon
        self.dropout_p = dropout
        self.gcn_hidden = gcn_hidden
        # If set, the head predicts a correction on top of the (scaled) last
        # observed demand -> much easier optimisation than predicting levels.
        self.residual_idx = residual_idx

        self.gcn1 = AttrGCNLayer(n_features, gcn_hidden, edge_attr_dim)
        self.gcn2 = AttrGCNLayer(gcn_hidden, gcn_hidden, edge_attr_dim)
        self.lstm = nn.LSTM(gcn_hidden, lstm_hidden, num_layers=2,
                            batch_first=True, dropout=dropout)
        self.head = nn.Linear(lstm_hidden, horizon)

    def freeze_gcn(self):
        """Random-GCN ablation: keep the random init, never train it."""
        for p in list(self.gcn1.parameters()) + list(self.gcn2.parameters()):
            p.requires_grad_(False)

    def encode_graph(self, x: torch.Tensor, adj: torch.Tensor,
                     edge_attr: torch.Tensor) -> torch.Tensor:
        """x: (B, T, N, F) -> (B, T, N, gcn_hidden) per-day graph embedding."""
        B, T, N, F_ = x.shape
        h = x.reshape(B * T, N, F_)
        h = self.gcn1(h, adj, edge_attr)
        h = F.dropout(h, p=self.dropout_p, training=self.training)
        h = self.gcn2(h, adj, edge_attr)
        return h.view(B, T, N, -1)

    def forward(self, x: torch.Tensor, adj: torch.Tensor,
                edge_attr: torch.Tensor) -> torch.Tensor:
        """x: (B, T=lookback, N, F) -> (B, N, horizon)"""
        g = self.encode_graph(x, adj, edge_attr)
        B, T, N, H = g.shape
        g = g.permute(0, 2, 1, 3).reshape(B * N, T, H)
        out, _ = self.lstm(g)
        out = F.dropout(out[:, -1, :], p=self.dropout_p, training=self.training)
        pred = self.head(out).view(B, N, self.horizon)
        if self.residual_idx is not None:
            pred = pred + x[:, -1, :, self.residual_idx].unsqueeze(-1)
        return pred

    @torch.no_grad()
    def embed(self, x: torch.Tensor, adj: torch.Tensor,
              edge_attr: torch.Tensor) -> torch.Tensor:
        """Deterministic GCN embedding of the issue day -> (B, N, gcn_hidden).
        This is each MARL agent's learned local observation."""
        was_training = self.training
        self.eval()
        g = self.encode_graph(x, adj, edge_attr)[:, -1]
        self.train(was_training)
        return g

    # ----------------------------------------------------------------- #
    # MC-Dropout predictive interval ("predictive", not "confidence")   #
    # ----------------------------------------------------------------- #
    @torch.no_grad()
    def predict_with_uncertainty(
        self, x: torch.Tensor, adj: torch.Tensor,
        edge_attr: torch.Tensor, n_samples: int = 50, alpha: float = 0.05,
        batch_size: int = 256,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Returns mean, lower (alpha/2), upper (1-alpha/2), each (B, N, H), on CPU."""
        dev = next(self.parameters()).device
        self.train()                                  # dropout ON at test time
        means, los, his = [], [], []
        for s in range(0, x.shape[0], batch_size):
            xb = x[s:s + batch_size].to(dev)
            preds = torch.stack([self(xb, adj, edge_attr) for _ in range(n_samples)])
            means.append(preds.mean(0).cpu())
            los.append(torch.quantile(preds, alpha / 2, dim=0).cpu())
            his.append(torch.quantile(preds, 1 - alpha / 2, dim=0).cpu())
        self.eval()
        return torch.cat(means), torch.cat(los), torch.cat(his)


# --------------------------------------------------------------------------- #
# Training utilities
# --------------------------------------------------------------------------- #
@torch.no_grad()
def predict_point(model: nn.Module, X: np.ndarray, adj, edge_attr,
                  batch_size: int = 256) -> np.ndarray:
    dev = next(model.parameters()).device
    model.eval()
    out = [model(torch.from_numpy(X[s:s + batch_size]).to(dev), adj, edge_attr).cpu()
           for s in range(0, len(X), batch_size)]
    return torch.cat(out).numpy()


@torch.no_grad()
def embed_windows(model: nn.Module, X: np.ndarray, adj, edge_attr,
                  batch_size: int = 256) -> np.ndarray:
    dev = next(model.parameters()).device
    out = [model.embed(torch.from_numpy(X[s:s + batch_size]).to(dev), adj, edge_attr).cpu()
           for s in range(0, len(X), batch_size)]
    return torch.cat(out).numpy()


def fit_forecaster(model: nn.Module, X_tr, y_tr, X_va, y_va, adj, edge_attr,
                   scaler: "PanelScaler", lr: float, batch_size: int,
                   epochs: int, patience: int, weight_decay: float = 1e-5,
                   trial=None, verbose: bool = False, seed: int = 0):
    """Adam + gradient clipping + early stopping on validation RMSE (in MW).
    Returns (model with best weights, best_val_rmse_mw, history)."""
    dev = next(model.parameters()).device
    g = torch.Generator().manual_seed(seed)
    ds = torch.utils.data.TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    loader = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True,
                                         generator=g, drop_last=len(ds) > batch_size)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=max(2, patience // 3))
    y_va_mw = scaler.inverse_y(y_va)
    best, best_state, bad, hist = math.inf, None, 0, []
    for epoch in range(epochs):
        model.train()
        tot = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(dev, non_blocking=True), yb.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = F.mse_loss(model(xb, adj, edge_attr), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            tot += loss.item() * len(xb)
        val_mw = scaler.inverse_y(predict_point(model, X_va, adj, edge_attr))
        score = float(np.sqrt(np.mean((y_va_mw - val_mw) ** 2)))
        sched.step(score)
        hist.append({"epoch": epoch, "train_mse_scaled": tot / len(ds), "val_rmse_mw": score})
        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f"epoch {epoch:3d}  train_mse(scaled) {tot/len(ds):.4f}  val RMSE {score:8.2f} MW")
        if score < best - 1e-3:
            best, bad = score, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
        if trial is not None:
            trial.report(score, epoch)
            if trial.should_prune():
                import optuna
                raise optuna.TrialPruned()
        if bad >= patience:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best, hist

## 5. MAPPO (9 decentralized actors + centralized critic)

In [29]:
"""
models/marl.py
==============
TRUE multi-agent PPO (MAPPO) under Centralized Training with Decentralized
Execution (CTDE).

    Actor_i(o_i)  ->  a_i         for i = 1..N   (decentralized execution)
    Critic(s)     ->  V(s)                        (centralized training only)

o_i = agent i's LOCAL observation (its own GCN embedding + scale-free local
      context), s = concatenation of all o_i.

Each agent has its own parameters and its own policy-gradient term; the
advantage comes from GAE(lambda) on the shared team reward using the
centralized critic, with a per-agent head for each agent's mixed
team / local reward (team_reward_weight in MARLConfig).

Fixes vs. the previous version
------------------------------
* Squashed-Gaussian policy: a = action_max * sigmoid(u), u ~ N(mu, sigma).
  The PRE-squash sample u is stored, so the PPO ratio is exact (previously the
  log-prob of the unclamped sample was compared with the log-prob of the
  clamped action, which biases the ratio).
* Real advantages: GAE with bootstrapping and episode-boundary masks
  (previously: normalised raw rewards, no baseline, dones ignored).
* Credit assignment: agent i optimises w*team + (1-w)*local_i with its own
  value head on the centralised critic. A pure shared team reward gave a
  216-dimensional action with one noisy scalar signal, and the policy
  drifted towards under-requesting in tests.
* Minibatch updates, gradient clipping, deterministic evaluation.

`share_parameters=True` is available for the ablation.
"""

from __future__ import annotations
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from typing import Callable, Dict, List, Optional, Tuple


# --------------------------------------------------------------------------- #
# Decentralized actor
# --------------------------------------------------------------------------- #
def rbf_basis(n_basis: int, n_hours: int = 24) -> np.ndarray:
    """(24, K) smooth radial-basis bumps spread over the day (K=0 -> identity)."""
    if not n_basis or n_basis >= n_hours:
        return np.eye(n_hours, dtype=np.float32)
    h = np.arange(n_hours)[:, None]
    c = np.linspace(0, n_hours - 1, n_basis)[None, :]
    w = n_hours / n_basis
    return np.exp(-0.5 * ((h - c) / w) ** 2).astype(np.float32)


class Actor(nn.Module):
    """Gaussian policy over the agent's hourly dispatch curve (fraction of the
    region's own forecast). u ~ N(mu(o), sigma) in K basis coordinates,
    curve = a_max * sigmoid(b0 + B u). The agent LEARNS when to dispatch /
    shed; no time slot is hand-picked."""
    def __init__(self, obs_dim: int, act_dim: int, hidden: int, action_max: float = 1.0,
                 action_init: float = 1.0, n_basis: int = 0, init_log_std: float = -1.0):
        super().__init__()
        self.action_max = action_max
        self.register_buffer("B", torch.from_numpy(rbf_basis(n_basis, act_dim)))
        k = self.B.shape[1]
        p0 = min(max(action_init / action_max, 1e-3), 1 - 1e-3)
        self.b0 = math.log(p0 / (1.0 - p0))       # initial request = action_init
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.mu = nn.Linear(hidden, k)
        nn.init.normal_(self.mu.weight, std=1e-3)
        nn.init.zeros_(self.mu.bias)
        self.log_std = nn.Parameter(torch.full((k,), float(init_log_std)))

    def forward(self, obs: torch.Tensor) -> Normal:
        mu = self.mu(self.net(obs))
        std = self.log_std.clamp(-5.0, 1.0).exp().expand_as(mu)
        return Normal(mu, std)

    def squash(self, u: torch.Tensor) -> torch.Tensor:
        return self.action_max * torch.sigmoid(self.b0 + u @ self.B.T)

    def act(self, obs: torch.Tensor, deterministic: bool = False):
        dist = self.forward(obs)
        u = dist.mean if deterministic else dist.sample()
        return self.squash(u), u, dist.log_prob(u).sum(-1)


# --------------------------------------------------------------------------- #
# Centralized critic (sees the global state)
# --------------------------------------------------------------------------- #
class CentralCritic(nn.Module):
    """Sees the global state; outputs one value per agent (each agent's
    mixed team/local return)."""
    def __init__(self, global_state_dim: int, hidden: int, n_agents: int = 1):
        super().__init__()
        self.v = nn.Sequential(
            nn.Linear(global_state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),           nn.Tanh(),
            nn.Linear(hidden, n_agents),
        )

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        return self.v(s)                                   # (..., N)


# --------------------------------------------------------------------------- #
# MAPPO container
# --------------------------------------------------------------------------- #
class MAPPO:
    """N decentralized actors + 1 centralized critic."""
    def __init__(self, cfg: MARLConfig, share_parameters: bool = False):
        self.cfg = cfg
        self.share = share_parameters
        mk = lambda: Actor(cfg.obs_dim_per_agent, cfg.action_dim_per_agent,
                           cfg.hidden_dim, cfg.action_max, cfg.action_init,
                           cfg.n_action_basis, cfg.init_log_std)
        if share_parameters:
            shared = mk()
            self.actors: List[Actor] = [shared] * cfg.n_agents
            actor_params = list(shared.parameters())
        else:
            self.actors = [mk() for _ in range(cfg.n_agents)]
            actor_params = [p for a in self.actors for p in a.parameters()]
        self.critic = CentralCritic(cfg.obs_dim_per_agent * cfg.n_agents, cfg.hidden_dim,
                                    cfg.n_agents)
        self.actor_params = actor_params
        self.actor_opt = torch.optim.Adam(actor_params, lr=cfg.lr_actor)
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr=cfg.lr_critic)

    # ------------------------------------------------------------------ #
    @torch.no_grad()
    def act_all(self, obs_per_agent: torch.Tensor, deterministic: bool = False):
        """obs_per_agent: (N, obs) or (B, N, obs).
        Returns actions (.., N, act) in [0, action_max], raw u, logp (.., N)."""
        acts, raws, logps = [], [], []
        for i, actor in enumerate(self.actors):
            a, u, lp = actor.act(obs_per_agent[..., i, :], deterministic)
            acts.append(a); raws.append(u); logps.append(lp)
        return (torch.stack(acts, -2), torch.stack(raws, -2), torch.stack(logps, -1))

    @torch.no_grad()
    def value(self, global_state: torch.Tensor) -> torch.Tensor:
        return self.critic(global_state)

    def state_dict(self) -> Dict:
        return {"actors": [copy.deepcopy(a.state_dict()) for a in self.actors],
                "critic": copy.deepcopy(self.critic.state_dict())}

    def load_state_dict(self, sd: Dict):
        for a, s in zip(self.actors, sd["actors"]):
            a.load_state_dict(s)
        self.critic.load_state_dict(sd["critic"])

    # ------------------------------------------------------------------ #
    # PPO update -- per-agent clipped surrogate, advantage from the
    # centralized critic (GAE), minibatched.
    # ------------------------------------------------------------------ #
    def update(self, batch: dict) -> Dict[str, float]:
        obs_i, raw_i = batch["obs_i"], batch["raw_i"]          # (B,N,o) (B,N,a)
        old_logp = batch["logp_i"]                             # (B,N)
        adv, ret, s = batch["adv"], batch["returns"], batch["global_state"]  # (B,N) (B,N)
        B = obs_i.shape[0]
        mb = min(self.cfg.minibatch_size, B)
        stats = {"pi_loss": 0.0, "v_loss": 0.0, "entropy": 0.0, "n": 0}
        for _ in range(self.cfg.n_epochs_per_update):
            perm = torch.randperm(B)
            for st in range(0, B, mb):
                idx = perm[st:st + mb]
                # ---- critic ----
                v_loss = F.mse_loss(self.critic(s[idx]), ret[idx])
                self.critic_opt.zero_grad()
                v_loss.backward()
                nn.utils.clip_grad_norm_(self.critic.parameters(), self.cfg.max_grad_norm)
                self.critic_opt.step()
                # ---- actors ----
                total, ent_sum = 0.0, 0.0
                for i, actor in enumerate(self.actors):
                    a_b = adv[idx, i]
                    dist = actor(obs_i[idx, i, :])
                    logp = dist.log_prob(raw_i[idx, i, :]).sum(-1)
                    ratio = torch.exp(logp - old_logp[idx, i])
                    surr = torch.min(ratio * a_b,
                                     torch.clamp(ratio, 1 - self.cfg.clip_ratio,
                                                 1 + self.cfg.clip_ratio) * a_b)
                    ent = dist.entropy().sum(-1).mean()
                    total = total - surr.mean() - self.cfg.entropy_coef * ent
                    ent_sum += ent.item()
                self.actor_opt.zero_grad()
                total.backward()
                nn.utils.clip_grad_norm_(self.actor_params, self.cfg.max_grad_norm)
                self.actor_opt.step()
                stats["pi_loss"] += total.item(); stats["v_loss"] += v_loss.item()
                stats["entropy"] += ent_sum / len(self.actors); stats["n"] += 1
        n = max(stats.pop("n"), 1)
        return {k: v / n for k, v in stats.items()}


# --------------------------------------------------------------------------- #
# Rollout / training loop (shared by tuning, final training, ablation, sweeps)
# --------------------------------------------------------------------------- #
def compute_gae(rews, vals, dones, last_val, gamma, lam):
    """Works for scalar or per-agent (T, N) rewards/values."""
    rews = np.asarray(rews, dtype=np.float32)
    vals = np.asarray(vals, dtype=np.float32)
    T = len(rews)
    adv = np.zeros_like(rews)
    last = 0.0
    for t in reversed(range(T)):
        next_v = last_val if t == T - 1 else vals[t + 1]
        nonterm = 1.0 - dones[t]
        delta = rews[t] + gamma * next_v * nonterm - vals[t]
        last = delta + gamma * lam * nonterm * last
        adv[t] = last
    return adv, adv + vals


def train_mappo(mappo: MAPPO, env, total_steps: int,
                eval_fn: Optional[Callable[[MAPPO], float]] = None,
                eval_every_updates: int = 10, verbose: bool = False) -> Dict:
    """Collect rollouts, update with GAE advantages; optionally keep the
    checkpoint with the best eval_fn score (higher is better)."""
    cfg = mappo.cfg
    obs, gs = env.reset()
    buf = {k: [] for k in ("obs", "gs", "raw", "logp", "r", "done", "v")}
    best_score, best_state, n_upd, ep_ret, ep_returns, log = -np.inf, None, 0, 0.0, [], []
    for step in range(total_steps):
        obs_t = torch.from_numpy(obs).float()
        gs_t = torch.from_numpy(gs).float()
        acts, raw, logp = mappo.act_all(obs_t)
        v = mappo.value(gs_t).numpy()                              # (N,)
        next_obs, next_gs, r, r_local, done, _ = env.step(acts.numpy())
        w = cfg.team_reward_weight
        r_agent = (w * r + (1.0 - w) * r_local).astype(np.float32)   # (N,)
        for k, val in zip(buf, (obs, gs, raw.numpy(), logp.numpy(), r_agent, float(done), v)):
            buf[k].append(val)
        ep_ret += r
        if done:
            ep_returns.append(ep_ret); ep_ret = 0.0
            obs, gs = env.reset()
        else:
            obs, gs = next_obs, next_gs

        if len(buf["r"]) >= cfg.rollout_len:
            last_v = (np.zeros(cfg.n_agents, np.float32) if done
                      else mappo.value(torch.from_numpy(gs).float()).numpy())
            adv, ret = compute_gae(buf["r"], buf["v"], buf["done"], last_v,
                                   cfg.gamma, cfg.gae_lambda)
            adv_t = torch.from_numpy(adv)
            batch = {
                "obs_i": torch.tensor(np.stack(buf["obs"]), dtype=torch.float32),
                "global_state": torch.tensor(np.stack(buf["gs"]), dtype=torch.float32),
                "raw_i": torch.tensor(np.stack(buf["raw"]), dtype=torch.float32),
                "logp_i": torch.tensor(np.stack(buf["logp"]), dtype=torch.float32),
                "adv": (adv_t - adv_t.mean(0)) / (adv_t.std(0) + 1e-8),
                "returns": torch.from_numpy(ret),
            }
            stats = mappo.update(batch)
            n_upd += 1
            for k in buf:
                buf[k].clear()
            if eval_fn is not None and n_upd % eval_every_updates == 0:
                score = eval_fn(mappo)
                log.append({"update": n_upd, "step": step + 1, "eval": score, **stats})
                if score > best_score:
                    best_score, best_state = score, mappo.state_dict()
                if verbose:
                    print(f"  update {n_upd:4d} step {step+1:7d}  eval {score:9.4f}  "
                          f"entropy {stats['entropy']:.2f}")
    if eval_fn is not None:
        score = eval_fn(mappo)
        if score > best_score:
            best_score, best_state = score, mappo.state_dict()
        mappo.load_state_dict(best_state)
    return {"best_eval": best_score, "episode_returns": ep_returns, "log": log}


@torch.no_grad()
def mappo_actions_batch(mappo: MAPPO, obs: np.ndarray) -> np.ndarray:
    """Deterministic actions for a batch of days: obs (D, N, o) -> (D, N, 24)."""
    a, _, _ = mappo.act_all(torch.from_numpy(obs).float(), deterministic=True)
    return a.numpy()

## 6. Multi-agent load-shedding environment

In [30]:
"""
env/load_shed_env.py
====================
Multi-agent load-shedding environment (one step = one day, 24 hourly slots).

Key design choices tied to user instructions and reviewer comments
------------------------------------------------------------------
1. NO HAND-TUNED SCHEDULING TIME. Each agent outputs a 24-dim curve of
   dispatch requests as a fraction of ITS OWN FORECAST (0 .. action_max).
   Where and when shedding happens is learned, never imposed.
2. TRUE MULTI-AGENT INTERFACE (Comment 1): step() takes (N, 24) actions and
   returns per-agent observations plus a global state.
3. LOAD-SHEDDING, not economic dispatch (Comment 10): the operator grants the
   requests subject to the HOURLY available capacity (pro-rata if the hour is
   short); anything the region actually needs beyond its grant is shed.
4. REWARD (Comment 8): scale-free terms, weights in config.RewardWeights.

Fixes vs. the previous version
------------------------------
* Requests were multiplied by the ACTUAL demand (unknown at decision time);
  they now scale the FORECAST and shedding is measured against the actual.
* Supply was a daily energy budget (energy could move between hours); it is
  now an hourly MW limit.
* Jain fairness was on absolute MWh (big regions look "unfair"); it is now on
  the served ratio of each region.
* The cost term penalised serving load; it now penalises over-dispatch
  (capacity reserved for a region that did not need it) in hours where
  capacity is binding -- reserving idle capacity is free.
* Observations now carry scale-free context (supply adequacy, own share,
  own forecast vs. recent level, own priority), so agents can see scarcity.
"""

from __future__ import annotations
import numpy as np
from dataclasses import dataclass
from typing import Dict, Optional, Tuple

N_EXTRA_OBS = 4          # adequacy, share, relative level, priority


def hourly_profile(grid: GridOpsConfig = CFG.grid) -> np.ndarray:
    """Typical BD daily load curve normalised so the evening peak = 1.0
    (the regional columns in version_5.xlsx are evening-peak MW).
    ASSUMPTION -- the dataset has no hourly resolution."""
    h = np.arange(24)
    base = grid.profile_base + grid.profile_day_amp * np.sin((h - 6) * np.pi / 12).clip(min=-0.6)
    evening = grid.profile_evening_amp * np.exp(-((h - grid.profile_peak_hour) ** 2) / 6.0)
    prof = base + evening
    return (prof / prof.max()).astype(np.float64)


HOURLY_PROFILE = hourly_profile()


@dataclass
class EnvSample:
    date: "pd.Timestamp"
    demand_forecast: np.ndarray   # (N, 24) from the GNN-LSTM (MW)
    demand_actual:   np.ndarray   # (N, 24) realised (MW)
    supply_cap_mw:   float        # available national capacity per hour (MW)
    gcn_embedding:   np.ndarray   # (N, E) learned per-region embedding
    trailing_mean:   np.ndarray   # (N,) recent 7-day mean peak demand (MW)
    supply_realized: Optional[np.ndarray] = None   # (N,) historical served peak MW
    is_stress: bool = False


# PCA compression of the GCN embedding inside the agents' observation
# (fitted on TRAINING-day embeddings in Section 16.4). With the full 64-128-dim
# embedding the agents memorised individual training days and generalised
# poorly to the shortage-heavy 2023-2024 period.
OBS_EMBED_PROJ = {"mean": None, "P": None}


def fit_obs_embedding_projection(emb_train: np.ndarray, k: int):
    """emb_train (S, N, E). k=0 -> embedding not observed (ablation)."""
    E = emb_train.shape[-1]
    X = emb_train.reshape(-1, E)
    mu = X.mean(0)
    if k <= 0:
        OBS_EMBED_PROJ.update(mean=mu, P=np.zeros((E, 0), np.float32))
        return 0.0
    U, S, Vt = np.linalg.svd(X - mu, full_matrices=False)
    P = Vt[:k].T / (S[:k] / np.sqrt(len(X)) + 1e-6)          # whitened
    OBS_EMBED_PROJ.update(mean=mu, P=P.astype(np.float32))
    return float((S[:k] ** 2).sum() / (S ** 2).sum())        # explained variance


def obs_dim_for(emb_dim: int) -> int:
    P = OBS_EMBED_PROJ["P"]
    return (emb_dim if P is None else P.shape[1]) + N_EXTRA_OBS


def build_obs_array(emb: np.ndarray, fc_peak: np.ndarray, cap: np.ndarray,
                    trailing: np.ndarray, prio: np.ndarray) -> np.ndarray:
    """Vectorised observation builder.
    emb (..., N, E), fc_peak (..., N), cap (...,), trailing (..., N) -> (..., N, k+4)"""
    if OBS_EMBED_PROJ["P"] is not None:
        emb = np.clip((emb - OBS_EMBED_PROJ["mean"]) @ OBS_EMBED_PROJ["P"], -5, 5)
    tot = fc_peak.sum(-1, keepdims=True)
    adequacy = np.clip(np.asarray(cap)[..., None] / np.maximum(tot, 1e-6), 0, 2) - 1.0
    adequacy = np.broadcast_to(adequacy, fc_peak.shape)
    share = fc_peak / np.maximum(tot, 1e-6) * fc_peak.shape[-1] - 1.0
    rel = np.clip(fc_peak / np.maximum(trailing, 1e-6) - 1.0, -1, 1)
    pr = np.broadcast_to(prio, fc_peak.shape)
    extra = np.stack([adequacy * 5.0, share, rel * 5.0, pr], axis=-1)
    return np.concatenate([emb, extra], axis=-1).astype(np.float32)


def apply_capacity(requested: np.ndarray, cap) -> np.ndarray:
    """Pro-rata curtailment of requests (…, N, 24) to the hourly capacity."""
    tot = requested.sum(-2, keepdims=True)                       # (…, 1, 24)
    cap = np.asarray(cap, dtype=np.float64)[..., None, None]
    scale = np.minimum(1.0, cap / np.maximum(tot, 1e-6))
    return requested * scale


def dispatch_outcome(demand_actual: np.ndarray, granted: np.ndarray, cap=None):
    """served / shed / over-dispatch. If cap is given, over-dispatch only counts
    in BINDING hours (grants exhaust capacity): reserving otherwise-idle
    capacity costs nothing, while reserving scarce capacity denies it to
    another region."""
    served = np.minimum(granted, demand_actual)
    shed = demand_actual - served
    over = granted - served
    if cap is not None:
        binding = granted.sum(-2, keepdims=True) >= np.asarray(cap)[..., None, None] * (1 - 1e-6)
        over = over * binding
    return served, shed, over


def _jain_fairness(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    s1, s2 = x.sum(), (x ** 2).sum()
    return 1.0 if s2 == 0 else float((s1 ** 2) / (len(x) * s2))


def compute_reward(demand_actual, served, shed, over, actions, prev_actions,
                   prio, w: RewardWeights):
    D_i = demand_actual.sum(1)                                   # (N,)
    D = max(D_i.sum(), 1e-6)
    shed_i = shed.sum(1)
    ens_pct = 100.0 * shed_i.sum() / D
    prio_pct = 100.0 * (prio * shed_i).sum() / max((prio * D_i).sum(), 1e-6)
    served_ratio = served.sum(1) / np.maximum(D_i, 1e-6)
    fair = _jain_fairness(served_ratio)
    over_pct = 100.0 * over.sum() / D
    smooth = 10.0 * np.abs(actions - prev_actions).mean() if prev_actions is not None else 0.0
    comps = {"ens": -w.alpha_ens * ens_pct,
             "priority": -w.beta_priority * prio_pct,
             "fairness": -w.gamma_fair * 100.0 * (1.0 - fair),
             "cost": -w.delta_cost * over_pct,
             "smooth": -w.epsilon_smooth * smooth}
    reward = float(sum(comps.values()))
    # LOCAL reward of agent i: its own shed and over-dispatch (% of its demand)
    Di = np.maximum(D_i, 1e-6)
    per_agent = (-(w.alpha_ens + w.beta_priority * prio) * 100.0 * shed_i / Di
                 - w.delta_cost * 100.0 * over.sum(1) / Di)
    if prev_actions is not None:
        per_agent = per_agent - w.epsilon_smooth * 10.0 * np.abs(actions - prev_actions).mean(1)
    return reward, per_agent, comps, fair


class LoadShedEnv:
    def __init__(self, dataset: list, reward_weights: RewardWeights | None = None,
                 seed: int = 0, prio: np.ndarray | None = None,
                 episode_len: int | None = None, random_start: bool = True,
                 stress_start_weight: float | None = None):
        self.data = dataset
        self.rw = reward_weights or CFG.reward
        self.rng = np.random.default_rng(seed)
        self.prio = PRIO_VEC.copy() if prio is None else np.asarray(prio, float)
        self.n_agents = len(REGIONS)
        self.episode_len = episode_len
        self.random_start = random_start
        self.t, self.end = 0, len(dataset)
        self.prev_actions = None
        # episode-start distribution: favour windows containing shortage days
        self._start_p = None
        n = len(dataset)
        if episode_len and n > episode_len:
            sw = CFG.marl.stress_start_weight if stress_start_weight is None else stress_start_weight
            stress = np.array([s.is_stress for s in dataset], float)
            cnt = np.convolve(stress, np.ones(episode_len), "valid")        # n-ep+1
            w = 1.0 + sw * cnt
            self._start_p = w / w.sum()

    def obs_of(self, s: EnvSample) -> np.ndarray:
        return build_obs_array(s.gcn_embedding, s.demand_forecast.max(1),
                               np.array(s.supply_cap_mw), s.trailing_mean, self.prio)

    def _observe(self, sample: EnvSample) -> Tuple[np.ndarray, np.ndarray]:
        obs = self.obs_of(sample)
        return obs, obs.reshape(-1)

    def reset(self, start: int | None = None) -> Tuple[np.ndarray, np.ndarray]:
        n = len(self.data)
        if start is None:
            if self.episode_len and self.random_start and n > self.episode_len:
                start = int(self.rng.choice(len(self._start_p), p=self._start_p))
            else:
                start = 0
        self.t = start
        self.end = min(n, start + self.episode_len) if self.episode_len else n
        self.prev_actions = None
        return self._observe(self.data[self.t])

    def step(self, actions: np.ndarray):
        """actions: (N, 24) requested fraction of forecast per region-hour."""
        s = self.data[self.t]
        actions = np.clip(actions, 0.0, None)
        granted = apply_capacity(actions * s.demand_forecast, s.supply_cap_mw)
        served, shed, over = dispatch_outcome(s.demand_actual, granted, s.supply_cap_mw)
        reward, per_agent, comps, fair = compute_reward(
            s.demand_actual, served, shed, over, actions, self.prev_actions,
            self.prio, self.rw)
        self.prev_actions = actions
        self.t += 1
        done = self.t >= self.end
        if done:
            next_obs = np.zeros((self.n_agents, self.obs_of(s).shape[-1]), np.float32)
            next_gs = next_obs.reshape(-1)
        else:
            next_obs, next_gs = self._observe(self.data[self.t])
        ens_i = shed.sum(1)
        info = {"date": s.date, "ens_total_mwh": float(ens_i.sum()),
                "ens_per_region_mwh": ens_i, "fairness_jain": fair,
                "priority_weighted_burden": float((self.prio * ens_i).sum()),
                "granted": granted, "served": served, "shed": shed, **comps}
        return next_obs, next_gs, reward, per_agent, done, info


def evaluate_on_env(mappo, env: LoadShedEnv) -> Dict:
    """Deterministic pass over every day of env.data (batched)."""
    obs = np.stack([env.obs_of(s) for s in env.data])                # (D, N, o)
    acts = mappo_actions_batch(mappo, obs)                           # (D, N, 24)
    fc = np.stack([s.demand_forecast for s in env.data])
    cap = np.array([s.supply_cap_mw for s in env.data])
    granted = apply_capacity(acts * fc, cap)
    rewards, prev = [], None
    for d, s in enumerate(env.data):
        served, shed, over = dispatch_outcome(s.demand_actual, granted[d], s.supply_cap_mw)
        r, *_ = compute_reward(s.demand_actual, served, shed, over, acts[d], prev,
                               env.prio, env.rw)
        rewards.append(r); prev = acts[d]
    return {"mean_reward": float(np.mean(rewards)), "granted": granted, "actions": acts}

## 7. Forecasting baselines (persistence, seasonal-naive, ridge, XGB)

In [31]:
"""
baselines/forecast_baselines.py
================================
Reviewer Comment 17 -- mandatory baseline hierarchy for forecasting.

1. Persistence     : y_hat[t+h] = y[t]
2. Seasonal-naive  : y_hat[t+h] = y[t+h-7]       (same weekday, last week)
3. Ridge regression on the issue-day features
4. XGBoost         (xgboost if available; RandomForest fallback)
5. LSTM-only       (identity graph)             -- ablation, Section 16.5
6. GCN-LSTM        (trained graph)              -- the proposed model

All baselines share the same leakage-safe features and train/val/test windows.
Fix: the old seasonal-naive used an HOURLY season (168) on DAILY data and
returned an empty slice whenever season == horizon.
"""

from __future__ import annotations
import numpy as np
from sklearn.linear_model import Ridge

try:
    from xgboost import XGBRegressor
    _HAS_XGB = True
except ImportError:
    from sklearn.ensemble import RandomForestRegressor
    _HAS_XGB = False


def persistence_forecast(y_hist: np.ndarray, horizon: int) -> np.ndarray:
    """y_hist: (T,) -> (horizon,) last observed value repeated."""
    return np.full(horizon, y_hist[-1], dtype=np.float32)


def seasonal_naive_forecast(y_hist: np.ndarray, horizon: int,
                            season: int = 7) -> np.ndarray:
    """Value of the same weekday one season ago (tiled if horizon > season)."""
    if len(y_hist) < season:
        return persistence_forecast(y_hist, horizon)
    last_season = y_hist[-season:]
    return np.array([last_season[h % season] for h in range(horizon)], dtype=np.float32)


def naive_baselines_from_windows(wd: Dict, split: str) -> Dict[str, np.ndarray]:
    """Persistence & seasonal-naive for every window -> (S, N, H) in MW."""
    o, H = wd["splits"][split], wd["horizon"]
    Y = wd["Y"]
    pers = np.repeat(Y[o][:, :, None], H, axis=2)
    seas = np.stack([np.stack([seasonal_naive_forecast(Y[:t + 1, i], H)
                               for i in range(Y.shape[1])]) for t in o])
    return {"Persistence": pers, "SeasonalNaive_7d": seas}


def tabular_from_windows(X: np.ndarray, y: np.ndarray | None = None):
    """Issue-day features per region + region one-hot, pooled over regions.
    X: (S, L, N, F) -> (S*N, F+N);  y: (S, N, H) -> (S*N, H)"""
    S, L, N, Fe = X.shape
    last = X[:, -1]                                          # (S, N, F)
    onehot = np.broadcast_to(np.eye(N, dtype=np.float32), (S, N, N))
    Xt = np.concatenate([last, onehot], axis=-1).reshape(S * N, Fe + N)
    return (Xt, None) if y is None else (Xt, y.reshape(S * N, -1))


class RidgeBaseline:
    def __init__(self, horizon: int, alpha: float = 1.0):
        self.horizon = horizon
        self.models = [Ridge(alpha=alpha) for _ in range(horizon)]

    def fit(self, X: np.ndarray, y: np.ndarray):
        for h in range(self.horizon):
            self.models[h].fit(X, y[:, h])
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.stack([m.predict(X) for m in self.models], axis=1)


class TreeBaseline:
    def __init__(self, horizon: int, seed: int = 0, **kw):
        self.horizon = horizon
        if _HAS_XGB:
            self.models = [XGBRegressor(n_estimators=400, max_depth=5,
                                        learning_rate=0.05, subsample=0.9,
                                        colsample_bytree=0.9, tree_method="hist",
                                        random_state=seed, n_jobs=-1, **kw)
                           for _ in range(horizon)]
        else:
            self.models = [RandomForestRegressor(n_estimators=300, max_depth=12,
                                                 n_jobs=-1, random_state=seed)
                           for _ in range(horizon)]

    def fit(self, X: np.ndarray, y: np.ndarray):
        for h in range(self.horizon):
            self.models[h].fit(X, y[:, h])
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.stack([m.predict(X) for m in self.models], axis=1)


def build_identity_graph(n_regions: int, device=None):
    """LSTM-only ablation: self-loops only -> no message passing."""
    adj = torch.eye(n_regions, device=device)
    edge_attr = torch.zeros(n_regions, n_regions, 2, device=device)
    return adj, edge_attr

## 8. Scheduling baselines (historical, proportional, priority, MILP oracle)

In [32]:
"""
baselines/schedule_baselines.py
================================
Reviewer Comment 7 -- MARL must beat simpler schedulers.

Every policy maps (demand_forecast (N,24), hourly capacity MW, priorities)
-> granted MW (N,24) that respects the hourly capacity. The environment then
measures shedding against the ACTUAL demand, exactly as for MAPPO.

1. HistoricalRealized     : what the grid actually did (served MW from the
                            dataset used as each region's hourly ceiling)
2. HistoricalShare        : capacity split by long-run demand shares
3. ProportionalToForecast : capacity split pro-rata to today's forecast
4. PriorityHeuristic      : fill high-priority regions first, then descend
5. GreedySmallestFirst    : fill the smallest regions first
6. LPOracle               : hourly LP with PERFECT FORESIGHT of actual demand,
                            minimising priority-weighted ENS (reference bound)

Fix: the old LP used one DAILY energy constraint, so energy could be shifted
between hours; capacity is now enforced hour by hour.
"""

from __future__ import annotations
import numpy as np
from scipy.optimize import linprog


def historical_realized(demand_actual: np.ndarray, supply_realized: np.ndarray) -> np.ndarray:
    """Regional served peak MW acts as that region's ceiling in every hour."""
    return np.minimum(demand_actual, supply_realized[:, None])


def historical_share(demand_forecast: np.ndarray, supply_mw: float,
                     historical_share_vec: np.ndarray) -> np.ndarray:
    alloc = np.outer(historical_share_vec, np.full(24, supply_mw))
    return apply_capacity(alloc, supply_mw)


def proportional_to_forecast(demand_forecast: np.ndarray, supply_mw: float) -> np.ndarray:
    tot = np.maximum(demand_forecast.sum(0, keepdims=True), 1e-6)
    return demand_forecast / tot * supply_mw


def _fill_in_order(demand_forecast, supply_mw, order):
    N, H = demand_forecast.shape
    alloc = np.zeros_like(demand_forecast)
    remaining = np.full(H, float(supply_mw))
    for idx in order:
        give = np.minimum(demand_forecast[idx], remaining)
        alloc[idx] = give
        remaining -= give
    # hand any leftover capacity out pro-rata (hedge against forecast error)
    tot = np.maximum(demand_forecast.sum(0), 1e-6)
    alloc += demand_forecast / tot * remaining
    return alloc


def priority_heuristic(demand_forecast: np.ndarray, supply_mw: float,
                       prio: np.ndarray = None) -> np.ndarray:
    prio = PRIO_VEC if prio is None else prio
    return _fill_in_order(demand_forecast, supply_mw, np.argsort(-prio))


def greedy_smallest_first(demand_forecast: np.ndarray, supply_mw: float) -> np.ndarray:
    return _fill_in_order(demand_forecast, supply_mw,
                          np.argsort(demand_forecast.sum(axis=1)))


def milp_optimum(demand_actual: np.ndarray, supply_mw: float,
                 prio: np.ndarray = None) -> np.ndarray:
    """Oracle: max sum_ih prio_i * x_ih  s.t. sum_i x_ih <= cap_h,
    0 <= x_ih <= demand_ih. (A plain LP -- integrality is not needed.)"""
    prio = PRIO_VEC if prio is None else prio
    N, H = demand_actual.shape
    c = -np.repeat(prio, H)
    A_ub = np.zeros((H, N * H))
    for h in range(H):
        A_ub[h, h::H] = 1.0
    b_ub = np.full(H, float(supply_mw))
    bounds = [(0.0, float(demand_actual[i, h])) for i in range(N) for h in range(H)]
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
    if not res.success:
        return proportional_to_forecast(demand_actual, supply_mw)
    return res.x.reshape(N, H)

## 9. Metrics: forecasting + intervals + scheduling + bootstrap CI

In [33]:
"""
evaluation/metrics.py
=====================
All metrics required by Reviewer Comments 4, 5, 18, 21.

Point-forecast metrics : MAE, RMSE, MAPE, sMAPE, R2, CV(RMSE)
Interval metrics       : PICP, MPIW, Winkler score
Scheduling metrics     : ENS, SAIDI (planned), Jain fairness,
                         Gini coefficient, priority-weighted burden,
                         worst-region burden, coefficient-of-variation
Statistical            : bootstrap confidence interval for any scalar metric
"""

from __future__ import annotations
import numpy as np
from typing import Callable, Dict, List, Tuple
from scipy import stats as _sps


# --------------------------------------------------------------------------- #
# Point-forecast metrics
# --------------------------------------------------------------------------- #
def mae(y, p):   return float(np.mean(np.abs(y - p)))
def rmse(y, p):  return float(np.sqrt(np.mean((y - p) ** 2)))

def mape(y, p, eps: float = 1e-6):
    return float(np.mean(np.abs((y - p) / np.maximum(np.abs(y), eps))) * 100)

def smape(y, p, eps: float = 1e-6):
    return float(np.mean(2 * np.abs(y - p) /
                         np.maximum(np.abs(y) + np.abs(p), eps)) * 100)

def r2(y, p):
    ss_res = np.sum((y - p) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return float(1.0 - ss_res / max(ss_tot, 1e-12))

def cv_rmse(y, p):
    return float(rmse(y, p) / max(np.mean(y), 1e-6) * 100)


def point_metrics(y, p) -> Dict[str, float]:
    return {"MAE": mae(y, p), "RMSE": rmse(y, p), "MAPE": mape(y, p),
            "sMAPE": smape(y, p), "R2": r2(y, p), "CV_RMSE_pct": cv_rmse(y, p)}


# --------------------------------------------------------------------------- #
# Interval metrics -- REPLACES "confidence interval" language
# --------------------------------------------------------------------------- #
def picp(y, lower, upper) -> float:
    """Prediction-interval coverage probability."""
    return float(np.mean((y >= lower) & (y <= upper)))

def mpiw(lower, upper) -> float:
    """Mean prediction-interval width."""
    return float(np.mean(upper - lower))

def winkler_score(y, lower, upper, alpha: float = 0.05) -> float:
    """Winkler / interval score. Lower is better."""
    width = upper - lower
    under = 2.0 / alpha * (lower - y) * (y < lower)
    over  = 2.0 / alpha * (y - upper) * (y > upper)
    return float(np.mean(width + under + over))


# --------------------------------------------------------------------------- #
# Scheduling / reliability metrics
# --------------------------------------------------------------------------- #
def ens(demand: np.ndarray, alloc: np.ndarray) -> float:
    """Energy not served (MWh)."""
    return float(np.maximum(demand - alloc, 0.0).sum())

def jain_fairness(x: np.ndarray) -> float:
    """Apply to SERVED RATIOS (served/demand) -- on absolute MWh the index
    simply penalises regions for being different sizes."""
    x = np.asarray(x, dtype=float); s1 = x.sum(); s2 = (x ** 2).sum()
    return 1.0 if s2 == 0 else float(s1 ** 2 / (len(x) * s2))

def gini(x: np.ndarray) -> float:
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x); cum = np.cumsum(x)
    if cum[-1] == 0: return 0.0
    return float((2 * np.sum((np.arange(1, n + 1) * x)) - (n + 1) * cum[-1])
                 / (n * cum[-1]))

def cv_of_burden(shed_per_region: np.ndarray) -> float:
    m = shed_per_region.mean()
    return float(shed_per_region.std() / max(m, 1e-6))

def priority_weighted_burden(shed_per_region: np.ndarray,
                             prio: np.ndarray) -> float:
    return float(np.sum(prio * shed_per_region))

def worst_region_burden(shed_per_region: np.ndarray) -> float:
    return float(shed_per_region.max())


# --------------------------------------------------------------------------- #
# Statistical: bootstrap confidence interval (Comment 18)
# --------------------------------------------------------------------------- #
def bootstrap_ci(values: np.ndarray, statistic: Callable = np.mean,
                 n_boot: int = 5000, alpha: float = 0.05,
                 seed: int = 0) -> Tuple[float, float, float]:
    """Returns (point_estimate, lower, upper) for a scalar statistic."""
    rng = np.random.default_rng(seed)
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return (np.nan, np.nan, np.nan)
    if len(vals) == 1:
        return (float(vals[0]),) * 3
    if statistic is np.mean:                          # vectorised fast path
        idx = rng.integers(0, len(vals), size=(n_boot, len(vals)))
        boot = vals[idx].mean(axis=1)
    else:
        boot = np.array([statistic(rng.choice(vals, size=len(vals), replace=True))
                         for _ in range(n_boot)])
    return (float(statistic(vals)),
            float(np.quantile(boot, alpha / 2)),
            float(np.quantile(boot, 1 - alpha / 2)))


def paired_bootstrap_diff(a: np.ndarray, b: np.ndarray, n_boot: int = 5000,
                          alpha: float = 0.05, seed: int = 0) -> Dict[str, float]:
    """Mean of (a - b) over paired days with bootstrap CI + Wilcoxon p-value."""
    d = np.asarray(a, float) - np.asarray(b, float)
    m, lo, hi = bootstrap_ci(d, n_boot=n_boot, alpha=alpha, seed=seed)
    try:
        p = float(_sps.wilcoxon(d[np.abs(d) > 1e-9]).pvalue) if (np.abs(d) > 1e-9).sum() > 10 else np.nan
    except ValueError:
        p = np.nan
    return {"mean_diff": m, "ci95_lo": lo, "ci95_hi": hi, "wilcoxon_p": p}


# --------------------------------------------------------------------------- #
# Scheduling evaluation over a list of EnvSample days
# --------------------------------------------------------------------------- #
PEAK_HOUR = int(np.argmax(HOURLY_PROFILE))


def schedule_day_table(samples: List["EnvSample"], granted: np.ndarray,
                       prio: np.ndarray) -> pd.DataFrame:
    """One row per day with every scheduling metric. granted: (D, N, 24)."""
    rows = []
    for d, s in enumerate(samples):
        served, shed, over = dispatch_outcome(s.demand_actual, granted[d], s.supply_cap_mw)
        D_i = s.demand_actual.sum(1); shed_i = shed.sum(1)
        rows.append({
            "date": s.date, "is_stress": s.is_stress,
            "ens_mwh": float(shed_i.sum()),
            "ens_pct": 100 * float(shed_i.sum() / D_i.sum()),
            "peak_shed_mw": float(shed[:, PEAK_HOUR].sum()),
            "priority_weighted_mwh": priority_weighted_burden(shed_i, prio),
            "jain_served_ratio": jain_fairness(served.sum(1) / D_i),
            "gini_shed": gini(shed_i),
            "worst_region_shed_pct": 100 * float((shed_i / D_i).max()),
            "over_dispatch_pct": 100 * float(over.sum() / D_i.sum()),
            **{f"shed_{r}_mwh": float(shed_i[i]) for i, r in enumerate(REGIONS)},
        })
    return pd.DataFrame(rows)


def summarize_schedule(day_df: pd.DataFrame, name: str) -> Dict[str, float]:
    out = {"policy": name, "days": len(day_df),
           "stress_days": int(day_df["is_stress"].sum())}
    for col in ["ens_mwh", "ens_pct", "peak_shed_mw", "priority_weighted_mwh",
                "jain_served_ratio", "worst_region_shed_pct", "over_dispatch_pct"]:
        m, lo, hi = bootstrap_ci(day_df[col].values, n_boot=2000)
        out[f"{col}_mean"], out[f"{col}_lo"], out[f"{col}_hi"] = m, lo, hi
    out["ens_total_gwh"] = day_df["ens_mwh"].sum() / 1e3
    reg = day_df[[f"shed_{r}_mwh" for r in REGIONS]].sum().values
    out["gini_region_shed_total"] = gini(reg)
    return out

## 10. Ablation grid runner

In [34]:
"""
evaluation/ablation.py
=======================
Reviewer Comment 2 -- the mandatory ablation grid.

Runs
----
    LSTM-only                (identity graph, no message passing)
    GNN-LSTM (trained GCN)
    GNN-LSTM (RANDOM GCN)    -- frozen random GCN: does graph LEARNING matter?
    LSTM + MAPPO
    GNN-LSTM + MAPPO         -- the full proposed system

Configurations without RL are scheduled with ProportionalToForecast so every
row has both forecasting AND scheduling metrics. Each configuration runs over
several seeds and is reported with mean, std and bootstrap 95% CI.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
import torch


ABLATION_CONFIGS = [
    {"name": "LSTM_only",           "use_graph": False, "train_gcn": True,  "use_rl": False},
    {"name": "GNN_LSTM_trained",    "use_graph": True,  "train_gcn": True,  "use_rl": False},
    {"name": "GNN_LSTM_random_GCN", "use_graph": True,  "train_gcn": False, "use_rl": False},
    {"name": "LSTM_PPO",            "use_graph": False, "train_gcn": True,  "use_rl": True},
    {"name": "GNN_LSTM_PPO_full",   "use_graph": True,  "train_gcn": True,  "use_rl": True},
]


def run_ablation(build_and_evaluate_fn, seeds=None, configs=ABLATION_CONFIGS):
    """
    build_and_evaluate_fn(config_dict, seed) -> dict of scalar metrics.
    Returns (summary DataFrame, per-seed raw DataFrame).
    """
    seeds = list(seeds or CFG.marl.seeds)
    rows, raw_rows = [], []
    for cfg in configs:
        per_seed = []
        for seed in seeds:
            t0 = time.time()
            metrics = build_and_evaluate_fn(cfg, seed)
            per_seed.append(metrics)
            raw_rows.append({"config": cfg["name"], "seed": seed, **metrics})
            print(f"  {cfg['name']:<22s} seed {seed:<4d} "
                  + "  ".join(f"{k}={v:.3f}" for k, v in metrics.items())
                  + f"   ({time.time()-t0:.0f}s)")
        df = pd.DataFrame(per_seed)
        summary = {"config": cfg["name"], "n_seeds": len(seeds)}
        for col in df.columns:
            vals = df[col].values
            mean, lo, hi = bootstrap_ci(vals)
            summary[f"{col}_mean"] = mean
            summary[f"{col}_std"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
            summary[f"{col}_ci95_lo"] = lo
            summary[f"{col}_ci95_hi"] = hi
        rows.append(summary)
    return pd.DataFrame(rows), pd.DataFrame(raw_rows)

## 11. Sensitivity analyses (reward, priority, economic)

In [35]:
"""
evaluation/sensitivity.py
==========================
Sensitivity analyses required by Reviewer Comments 8, 15, 19.

reward_weight_sweep()   -- one-at-a-time (default) or full-factorial
priority_score_sweep()  -- uniform scaling AND independent per-region
                           Monte-Carlo perturbations (+/-20%)
economic_sweep()        -- VOLL, implementation cost, discount rate, ENS scale

Fixes vs. the previous version
------------------------------
* A uniform +/-20% scaling of ALL priorities leaves every ranking -- and
  therefore every priority-driven schedule -- unchanged; it can only rescale
  the metric. Independent per-region draws are what actually test robustness.
* The full 3^5 = 243-point reward grid needs 243 MAPPO trainings; the default
  is now a one-at-a-time sweep (2*5 + 1 = 11 trainings).
* The weight-name mapping no longer relies on chained str.replace().
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from itertools import product


def reward_weight_sweep(evaluate_fn, perturbations=(-0.2, 0.2), mode: str = "oat"):
    """evaluate_fn(RewardWeights) -> dict of scheduling metrics."""
    base = CFG.reward.as_dict()
    keys = list(base.keys())
    if mode == "full":
        combos = list(product((-0.2, 0.0, 0.2), repeat=len(keys)))
    else:
        combos = [tuple(0.0 for _ in keys)]
        for j in range(len(keys)):
            for d in perturbations:
                c = [0.0] * len(keys); c[j] = d
                combos.append(tuple(c))
    rows = []
    for combo in combos:
        rw = RewardWeights.from_short({k: base[k] * (1 + d) for k, d in zip(keys, combo)})
        m = evaluate_fn(rw)
        rows.append({**{f"d_{k}": d for k, d in zip(keys, combo)}, **m})
    return pd.DataFrame(rows)


def priority_score_sweep(evaluate_fn, perturbations=(-0.2, -0.1, 0.0, 0.1, 0.2),
                         n_random: int = 200, max_rel: float = 0.2, seed: int = 0):
    """evaluate_fn(priority_vector (N,)) -> dict of scheduling metrics."""
    rows = []
    for delta in perturbations:
        p = np.clip(PRIO_VEC * (1 + delta), 0.0, 1.0)
        rows.append({"kind": "uniform", "priority_delta": delta,
                     "rank_changed": False, **evaluate_fn(p)})
    rng = np.random.default_rng(seed)
    base_rank = np.argsort(-PRIO_VEC)
    for k in range(n_random):
        p = np.clip(PRIO_VEC * (1 + rng.uniform(-max_rel, max_rel, len(PRIO_VEC))), 0, 1)
        rows.append({"kind": "per_region_mc", "priority_delta": np.nan, "draw": k,
                     "rank_changed": bool((np.argsort(-p) != base_rank).any()),
                     **{f"prio_{r}": p[i] for i, r in enumerate(REGIONS)},
                     **evaluate_fn(p)})
    return pd.DataFrame(rows)


def economic_sweep(annual_ens_reduction_mwh: float, evaluate_econ_fn):
    """Vary VOLL, implementation cost, discount rate, and ENS-reduction fraction."""
    voll_grid = [22.5, 45.0, 90.0]          # BDT/kWh  (0.5x, 1x, 2x)
    cost_grid = [0.25e9, 0.50e9, 1.0e9, 2.0e9]
    disc_grid = [0.06, 0.10, 0.14]
    ens_scale = [0.5, 1.0]                  # 50% and 100% of measured reduction
    rows = []
    for v, c, d, s in product(voll_grid, cost_grid, disc_grid, ens_scale):
        econ = EconomicConfig(voll_bdt_per_kwh=v, implementation_cost_bdt=c,
                              discount_rate=d)
        metrics = evaluate_econ_fn(econ, annual_ens_reduction_mwh * s)
        rows.append({"voll_bdt_per_kwh": v, "impl_cost_bdt": c,
                     "discount_rate": d, "ens_scale": s, **metrics})
    return pd.DataFrame(rows)


def economic_metrics(econ: EconomicConfig, annual_ens_reduction_mwh: float) -> dict:
    """NPV, BCR, simple payback for one economic assumption bundle."""
    annual_saving = annual_ens_reduction_mwh * 1000 * econ.voll_bdt_per_kwh
    discount = (1 + econ.discount_rate) ** np.arange(1, econ.horizon_years + 1)
    npv = float(np.sum((annual_saving - econ.annual_opex_bdt) / discount)
                - econ.implementation_cost_bdt)
    total_benefits = float(np.sum(annual_saving / discount))
    total_costs = float(econ.implementation_cost_bdt + np.sum(econ.annual_opex_bdt / discount))
    bcr = total_benefits / max(total_costs, 1.0)
    net = annual_saving - econ.annual_opex_bdt
    payback = econ.implementation_cost_bdt / net if net > 0 else np.inf
    return {"NPV_BDT": npv, "BCR": bcr, "payback_years": payback}

## 12. Optuna Bayesian tuning -- forecaster

In [36]:
"""
tuning/optuna_forecaster.py
============================
Bayesian (TPE) hyper-parameter tuning of the GNN-LSTM forecaster.

Search space
------------
gcn_hidden   : {32, 64, 128}
lstm_hidden  : {64, 128, 256}
dropout      : Uniform(0.1, 0.4)
lr           : LogUniform(1e-4, 3e-3)
batch_size   : {32, 64, 128}
weight_decay : LogUniform(1e-6, 1e-3)

Objective : validation RMSE in MW (inverse-scaled), with early stopping
Pruner    : MedianPruner

Names are prefixed (build_forecaster_objective / run_forecaster_tuning) so
the MAPPO tuning cell no longer overwrites them.
"""

from __future__ import annotations
import numpy as np
import torch
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

optuna.logging.set_verbosity(optuna.logging.WARNING)


def build_forecaster_objective(X_train, y_train, X_val, y_val, adj, edge_attr,
                               scaler, n_features, n_regions, lookback, horizon,
                               max_epochs: int = 20, patience: int = 6):
    def objective(trial: optuna.Trial) -> float:
        params = {
            "gcn_hidden": trial.suggest_categorical("gcn_hidden", [32, 64, 128]),
            "lstm_hidden": trial.suggest_categorical("lstm_hidden", [64, 128, 256]),
            "dropout": trial.suggest_float("dropout", 0.1, 0.4),
            "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        }
        seed_everything(1000 + trial.number)
        model = GNNLSTMForecaster(n_features, n_regions, lookback, horizon,
                                  gcn_hidden=params["gcn_hidden"],
                                  lstm_hidden=params["lstm_hidden"],
                                  dropout=params["dropout"],
                                  residual_idx=scaler.residual_idx).to(DEVICE)
        _, best, _ = fit_forecaster(model, X_train, y_train, X_val, y_val, adj,
                                    edge_attr, scaler, lr=params["lr"],
                                    batch_size=params["batch_size"],
                                    epochs=max_epochs, patience=patience,
                                    weight_decay=params["weight_decay"],
                                    trial=trial, seed=trial.number)
        return best
    return objective


def run_forecaster_tuning(objective, n_trials: int = 40, seed: int = 42):
    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=seed),
                                pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=5))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study

## 13. Optuna Bayesian tuning -- MAPPO

In [37]:
"""
tuning/optuna_marl.py
======================
Bayesian (TPE) tuning of the MAPPO learner on the REAL environment
(training days for learning, validation days for the score).

Search space
------------
lr_actor      : LogUniform(1e-5, 1e-3)
lr_critic     : LogUniform(1e-4, 3e-3)
gamma         : Uniform(0.0, 0.95)
gae_lambda    : Uniform(0.0, 0.95)
n_action_basis: {4, 6, 8}
obs_embed_dim : {0, 2, 4}  (PCA dims of the GCN embedding in the observation)
clip_ratio    : Uniform(0.1, 0.3)
entropy_coef  : LogUniform(1e-5, 1e-2)
hidden_dim    : {64, 128, 256}
team_reward_w : Uniform(0.5, 1.0)   (team vs local credit mix)

Objective : mean daily team reward on the validation environment,
            deterministic policy, averaged over seeds.
"""

from __future__ import annotations
import numpy as np
import torch
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from dataclasses import replace


def evaluate_policy(mappo: MAPPO, env) -> float:
    """Deterministic mean daily reward over every day of env.data."""
    return evaluate_on_env(mappo, env)["mean_reward"]


def marl_cfg_from_params(base_cfg: MARLConfig, params: dict, **overrides) -> MARLConfig:
    keep = {k: v for k, v in params.items() if hasattr(base_cfg, k)}
    return replace(base_cfg, **keep, **overrides)


def build_marl_objective(train_env_factory, val_env, base_cfg: MARLConfig,
                         seeds, train_steps: int = 20_000, emb_train=None):
    """train_env_factory(seed) -> LoadShedEnv on training days."""
    def objective(trial: optuna.Trial) -> float:
        params = {
            "hidden_dim": trial.suggest_categorical("hidden_dim", [64, 128, 256]),
            "lr_actor": trial.suggest_float("lr_actor", 1e-5, 1e-3, log=True),
            "lr_critic": trial.suggest_float("lr_critic", 1e-4, 3e-3, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 0.95),
            "gae_lambda": trial.suggest_float("gae_lambda", 0.0, 0.95),
            "clip_ratio": trial.suggest_float("clip_ratio", 0.1, 0.3),
            "entropy_coef": trial.suggest_float("entropy_coef", 1e-5, 1e-2, log=True),
            "team_reward_weight": trial.suggest_float("team_reward_weight", 0.5, 1.0),
            "n_action_basis": trial.suggest_categorical("n_action_basis", [4, 6, 8]),
        }
        overrides = {"total_env_steps": train_steps}
        if emb_train is not None:
            params["obs_embed_dim"] = trial.suggest_categorical("obs_embed_dim", [0, 2, 4])
            fit_obs_embedding_projection(emb_train, params["obs_embed_dim"])
            overrides["obs_dim_per_agent"] = obs_dim_for(emb_train.shape[-1])
        cfg = marl_cfg_from_params(base_cfg, params, **overrides)
        scores = []
        for s_idx, seed in enumerate(seeds):
            seed_everything(seed)
            mappo = MAPPO(cfg)
            res = train_mappo(mappo, train_env_factory(seed), train_steps,
                              eval_fn=lambda m: evaluate_policy(m, val_env),
                              eval_every_updates=max(1, train_steps // cfg.rollout_len // 4))
            scores.append(res["best_eval"])
            trial.report(float(np.mean(scores)), step=s_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(scores))
    return objective


def run_marl_tuning(objective, n_trials: int = 25, seed: int = 42):
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=seed),
                                pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=0))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study

## 14. Scenario projection 2025-2040 (low / central / high)

In [38]:
"""
forecasting/scenario_2025_2040.py
==================================
Long-horizon SCENARIO PROJECTION for 2025-01-01 .. 2040-12-31.

Reviewer framing (Comment 13): extrapolating 15+ years with a CAGR is not a
validated forecast, so this module
    - produces low / central / high growth scenarios,
    - calibrates seasonality, weekend effect and weather climatology from the
      REAL 2019-2024 data (previously hard-coded, and the weekend discount
      was applied twice -- once daily and again in the hourly shape),
    - propagates the forecaster's measured error (test MAPE) as a
      Monte-Carlo predictive band,
    - pairs demand with an explicit capacity-expansion supply scenario
      (previously capacity stayed flat at the 2019-2024 mean for 16 years),
    - is clearly labelled "scenario projection", not "forecast".

Output: long hourly frame (date, region, hour, demand_mean, demand_lo90,
demand_hi90, scenario) -- regional values are evening-peak-consistent MW.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List


@dataclass
class ScenarioAssumptions:
    """Every assumption a reviewer might question is EXPOSED here."""
    cagr_low:     float = 0.045   # 4.5%  ("policy-conservative")
    cagr_central: float = 0.065   # 6.5%  (BPDB Vision 2041 mid-range)
    cagr_high:    float = 0.085   # 8.5%  ("industrial-boom")
    regional_multipliers: Dict[str, float] = None
    monthly_factors: Dict[str, Dict[int, float]] = None   # region -> month -> factor
    weekend_discount: float = 0.95
    lstm_mape: float = 0.06        # overwritten with the measured test MAPE
    n_mc_paths: int = 200
    anchor_date: str = "2024-07-01"   # mid-point of the base year
    supply_cagr: float = 0.060
    supply_base_mw: float = None      # set by calibrate_from_history

    def __post_init__(self):
        if self.regional_multipliers is None:
            self.regional_multipliers = {
                "Dhaka": 1.10, "Chattogram": 1.15, "Sylhet": 0.95,
                "Khulna": 0.95, "Rajshahi": 0.95, "Cumilla": 1.00,
                "Mymensingh": 0.95, "Rangpur": 0.90, "Barishal": 0.85,
            }
        if self.monthly_factors is None:
            default = {1: 0.82, 2: 0.85, 3: 0.95, 4: 1.10, 5: 1.18, 6: 1.20,
                       7: 1.15, 8: 1.10, 9: 1.05, 10: 0.98, 11: 0.90, 12: 0.83}
            self.monthly_factors = {r: dict(default) for r in REGIONS}


def calibrate_from_history(realized: pd.DataFrame, raw: pd.DataFrame,
                           assumptions: ScenarioAssumptions,
                           years=(2022, 2023, 2024)) -> Dict:
    """Fill monthly factors, weekend discount, supply base and weather /
    holiday climatology from the real data. Returns the climatology dict."""
    r = realized[~realized["is_imputed"].astype(bool)].copy()
    r["year"] = r["timestamp"].dt.year
    r["month"] = r["timestamp"].dt.month
    r = r[r["year"].isin(years)]
    # monthly factor = month mean / annual mean, per region, averaged over years
    ym = r.groupby(["region", "year", "month"])["demand_mw"].mean()
    yr = r.groupby(["region", "year"])["demand_mw"].mean()
    ratio = (ym / yr.reindex(ym.index.droplevel("month")).values).groupby(
        ["region", "month"]).mean()
    mf = {reg: {m: float(ratio.loc[(reg, m)]) for m in range(1, 13)} for reg in REGIONS}
    for reg in REGIONS:                                # renormalise to mean 1
        s = float(np.mean(list(mf[reg].values())))
        mf[reg] = {m: float(v / s) for m, v in mf[reg].items()}
    assumptions.monthly_factors = mf
    # weekend discount: weekend mean / weekday mean within the same month
    r["wk"] = r["timestamp"].dt.dayofweek.isin([4, 5])
    g = r.groupby(["region", "year", "month", "wk"])["demand_mw"].mean().unstack("wk")
    assumptions.weekend_discount = float((g[True] / g[False]).mean())
    # supply base: demonstrated national served peak in the last real year
    last_year = int(r["year"].max())
    nat = (r[r["year"] == last_year].groupby("timestamp")["supply_mw_realized"].sum())
    assumptions.supply_base_mw = float(nat.quantile(CFG.grid.supply_base_quantile))
    # weather climatology (for the projection's decision-time features)
    t = raw[[RAW_DATE_COL, "Maximum Temperature in Dhaka was"]].copy()
    t["doy"] = pd.to_datetime(t[RAW_DATE_COL]).dt.dayofyear.clip(upper=365)
    clim = (t.groupby("doy")["Maximum Temperature in Dhaka was"].mean()
             .reindex(range(1, 366)).interpolate().bfill().ffill())
    clim = clim.rolling(15, center=True, min_periods=1).mean()
    # fixed-date holidays that recur in >= 3 different years
    h = raw[[RAW_DATE_COL, "Holiday_cat"]].copy()
    h["md"] = pd.to_datetime(h[RAW_DATE_COL]).dt.strftime("%m-%d")
    h["yr"] = pd.to_datetime(h[RAW_DATE_COL]).dt.year
    hh = h[h["Holiday_cat"] > 0].groupby("md").agg(n=("yr", "nunique"),
                                                    cat=("Holiday_cat", lambda s: s.mode().iat[0]))
    fixed = hh[hh["n"] >= 3]["cat"].astype(int).to_dict()
    return {"temp_by_doy": clim.to_dict(), "fixed_holidays": fixed}


def growth_factor(dates: pd.DatetimeIndex, assumptions: ScenarioAssumptions,
                  scenario: str) -> np.ndarray:
    cagr = {"low": assumptions.cagr_low, "central": assumptions.cagr_central,
            "high": assumptions.cagr_high}[scenario]
    yrs = ((dates - pd.Timestamp(assumptions.anchor_date)).days / 365.25).values
    mult = np.array([assumptions.regional_multipliers[r] for r in REGIONS])
    return (1.0 + cagr * mult[None, :]) ** yrs[:, None]            # (D, N)


def project_daily(base_year_mean_mw: Dict[str, float], assumptions: ScenarioAssumptions,
                  scenario: str = "central", start: str = "2025-01-01",
                  end: str = "2040-12-31", seed: int = 0) -> Dict[str, np.ndarray]:
    """Daily peak MW per region: mean and 90% band -> dict of (D, N) arrays."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, end, freq="D")
    base = np.array([base_year_mean_mw[r] for r in REGIONS])
    months = dates.month.values
    mf = np.array([[assumptions.monthly_factors[r][m] for r in REGIONS] for m in months])
    wk = np.where(dates.dayofweek.isin([4, 5]), assumptions.weekend_discount, 1.0)
    seasonal = base[None, :] * mf * wk[:, None]                    # growth-free shape
    mean = seasonal * growth_factor(dates, assumptions, scenario)
    rel = rng.normal(1.0, assumptions.lstm_mape,
                     size=(len(dates), len(REGIONS), assumptions.n_mc_paths))
    lo = mean * np.quantile(rel, 0.05, axis=-1)
    hi = mean * np.quantile(rel, 0.95, axis=-1)
    return {"dates": dates, "mean": mean, "lo90": lo, "hi90": hi, "seasonal": seasonal}


def project_scenario(base_year_mean_mw: Dict[str, float], assumptions: ScenarioAssumptions,
                     scenario: str = "central", start: str = "2025-01-01",
                     end: str = "2040-12-31", seed: int = 0) -> pd.DataFrame:
    """Long hourly frame (vectorised): date, region, hour, demand_mean,
    demand_lo90, demand_hi90, scenario."""
    p = project_daily(base_year_mean_mw, assumptions, scenario, start, end, seed)
    D, N, H = len(p["dates"]), len(REGIONS), 24
    prof = HOURLY_PROFILE[None, None, :]
    return pd.DataFrame({
        "date": np.repeat(p["dates"].values, N * H),
        "region": pd.Categorical(np.tile(np.repeat(REGIONS, H), D), categories=REGIONS),
        "hour": np.tile(np.arange(H), D * N).astype(np.int8),
        "demand_mean": (p["mean"][..., None] * prof).reshape(-1).astype(np.float32),
        "demand_lo90": (p["lo90"][..., None] * prof).reshape(-1).astype(np.float32),
        "demand_hi90": (p["hi90"][..., None] * prof).reshape(-1).astype(np.float32),
        "scenario": scenario,
    })


def project_all_scenarios(base_year_mean_mw: Dict[str, float],
                          assumptions: ScenarioAssumptions | None = None,
                          start: str = "2025-01-01", end: str = "2040-12-31"
                          ) -> pd.DataFrame:
    assumptions = assumptions or ScenarioAssumptions()
    frames = [project_scenario(base_year_mean_mw, assumptions, s, start, end, seed=i)
              for i, s in enumerate(("low", "central", "high"))]
    return pd.concat(frames, ignore_index=True)


def project_supply(dates, assumptions: ScenarioAssumptions) -> np.ndarray:
    """Available national capacity per hour (MW) under the supply scenario."""
    dates = pd.DatetimeIndex(dates)
    yrs = ((dates - pd.Timestamp(assumptions.anchor_date)).days / 365.25).values
    return assumptions.supply_base_mw * (1.0 + assumptions.supply_cagr) ** yrs


def build_projection_embeddings(model, raw: pd.DataFrame, seasonal_mw: np.ndarray,
                                proj_dates: pd.DatetimeIndex, clim: Dict,
                                scaler: "PanelScaler", adj, edge_attr,
                                cfg: DataConfig = CFG.data) -> np.ndarray:
    """GCN embeddings for every projected day, i.e. what each MARL agent would
    observe. The encoder is fed a GROWTH-DEFLATED demand path (2024 level with
    projected seasonality) plus climatological temperature and fixed-date
    holidays, so its inputs stay inside the training distribution; scarcity
    itself reaches the agents through the scale-free observation extras."""
    L = cfg.lookback_days
    fixed = clim["fixed_holidays"]
    synth = pd.DataFrame({RAW_DATE_COL: proj_dates})
    doy = np.minimum(np.asarray(proj_dates.dayofyear), 365)
    synth["Maximum Temperature in Dhaka was"] = [clim["temp_by_doy"][int(d)] for d in doy]
    synth["Holiday_cat"] = [fixed.get(d.strftime("%m-%d"), 0) for d in proj_dates]
    synth["Holiday name"] = np.where(synth["Holiday_cat"] > 0, "Projected fixed-date holiday",
                                     "No Holiday")
    for i, r in enumerate(REGIONS):
        synth[f"{r}_demand"] = seasonal_mw[:, i]
    keep = list(synth.columns)
    hist = raw[keep].tail(L + 60)
    combo = _regularize_daily(pd.concat([hist, synth], ignore_index=True),
                              cfg.max_interp_gap_days)
    feats_p = build_leakage_safe_features(combo, cfg, require_realized=False)
    F, _, _, dates = panel_arrays(feats_p, list(cfg.observed_at_decision_time))
    pos = dates.get_indexer(proj_dates - pd.Timedelta(days=1))    # issue day
    if (pos < L - 1).any():
        raise ValueError("Not enough history before the projection start.")
    emb = []
    for s in range(0, len(pos), 512):
        Xb = scaler.transform_x(gather_windows(F, pos[s:s + 512], L)).astype(np.float32)
        emb.append(embed_windows(model, Xb, adj, edge_attr))
    return np.concatenate(emb)                                       # (D, N, E)

## 15. Daily per-region schedule + Markdown report

In [39]:
"""
reports/daily_scheduler_report.py
==================================
"provide clear scheduling of every day for each of the regions with a
 detailed report"

For each day 2025-01-01 .. 2040-12-31 and each of the nine regions:
    - projected demand (MWh/day, peak MW)
    - MARL dispatch (MWh allocated) and load shed (MWh, peak-hour MW)
    - LEARNED shedding window (first/last hour, hours list, count)
    - reason code: supply-constrained / allocation-limited / none
and an hourly (24 x allocated, 24 x shed) schedule per region-day.

Outputs (central scenario keeps the original file names)
-------
    outputs/daily_schedule_2025_2040.csv          (+ _low / _high variants)
    outputs/daily_hourly_schedule_2025_2040_<scenario>.csv.gz
    outputs/daily_schedule_summary.md             (all scenarios)
    outputs/per_region/<region>_yearly.csv        (+ _low / _high variants)

Fix: the old version filtered the full projection frame once per date
(O(days^2), hours for 5,844 days) -- now fully vectorised.
"""

from __future__ import annotations
import os
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Callable, Dict


def _hours_to_ranges(hours: np.ndarray) -> str:
    if hours.size == 0:
        return ""
    out, start, prev = [], hours[0], hours[0]
    for h in hours[1:]:
        if h != prev + 1:
            out.append(f"{start:02d}-{prev + 1:02d}"); start = h
        prev = h
    out.append(f"{start:02d}-{prev + 1:02d}")
    return ";".join(out)


def generate_daily_report(
    projected_demand_df: pd.DataFrame,   # hourly frame from scenario_2025_2040
    policy_fn: Callable,                 # policy_fn(demand (D,N,24), cap (D,), dates) -> (D,N,24)
    supply_fn: Callable,                 # supply_fn(dates) -> (D,) hourly capacity MW
    output_dir: str = "./outputs",
    scenario: str = "central",
    write_hourly: bool = True,
) -> pd.DataFrame:
    out_dir = Path(output_dir)
    (out_dir / "per_region").mkdir(parents=True, exist_ok=True)
    suffix = "" if scenario == "central" else f"_{scenario}"

    df = projected_demand_df.sort_values(["date", "region", "hour"])
    dates = pd.DatetimeIndex(pd.unique(df["date"]))
    D, N = len(dates), len(REGIONS)
    demand = df["demand_mean"].to_numpy(np.float64).reshape(D, N, 24)
    cap = np.asarray(supply_fn(dates), dtype=np.float64)
    granted = apply_capacity(policy_fn(demand, cap, dates), cap)
    served, shed, over = dispatch_outcome(demand, granted)

    national_short = demand.sum(1) > cap[:, None] + 1e-6          # (D, 24)
    has_shed = shed > 1e-3                                         # (D, N, 24)
    any_shed = has_shed.any(-1)
    constrained = (has_shed & national_short[:, None, :]).any(-1)
    reason = np.where(~any_shed, "none",
                      np.where(constrained, "supply-constrained", "allocation-limited"))
    first = np.where(any_shed, has_shed.argmax(-1), -1)
    last = np.where(any_shed, 23 - has_shed[..., ::-1].argmax(-1), -1)
    windows = [[_hours_to_ranges(np.where(has_shed[d, i])[0]) for i in range(N)]
               for d in range(D)]

    out = pd.DataFrame({
        "date": np.repeat(dates.date, N),
        "scenario": scenario,
        "region": np.tile(REGIONS, D),
        "priority_weight": np.tile(PRIO_VEC, D),
        "demand_mwh": demand.sum(-1).reshape(-1),
        "peak_hour_demand_mw": demand.max(-1).reshape(-1),
        "supply_cap_mw": np.repeat(cap, N),
        "allocated_mwh": served.sum(-1).reshape(-1),
        "shed_mwh": shed.sum(-1).reshape(-1),
        "shed_pct": (100 * shed.sum(-1) / np.maximum(demand.sum(-1), 1e-6)).reshape(-1),
        "peak_hour_shed_mw": shed.max(-1).reshape(-1),
        "shed_window_start_hour": first.reshape(-1),
        "shed_window_end_hour": last.reshape(-1),
        "shed_hours_count": has_shed.sum(-1).reshape(-1),
        "shed_hours": [w for row in windows for w in row],
        "reason_code": reason.reshape(-1),
    })
    out.to_csv(out_dir / f"daily_schedule_2025_2040{suffix}.csv", index=False)

    if write_hourly:
        hourly = out[["date", "region"]].copy()
        a = served.reshape(D * N, 24); s = shed.reshape(D * N, 24)
        for h in range(24):
            hourly[f"alloc_mw_h{h:02d}"] = a[:, h].round(1)
        for h in range(24):
            hourly[f"shed_mw_h{h:02d}"] = s[:, h].round(1)
        hourly.to_csv(out_dir / f"daily_hourly_schedule_2025_2040_{scenario}.csv.gz",
                      index=False, compression="gzip")

    out["year"] = pd.to_datetime(out["date"]).dt.year
    for region in REGIONS:
        sub = out[out["region"] == region]
        yearly = (sub.groupby("year")
                     .agg(total_demand_mwh=("demand_mwh", "sum"),
                          total_shed_mwh=("shed_mwh", "sum"),
                          shed_days=("shed_mwh", lambda x: int((x > 1e-3).sum())),
                          avg_daily_shed_hours=("shed_hours_count", "mean"),
                          worst_day_shed_mwh=("shed_mwh", "max"),
                          worst_peak_shed_mw=("peak_hour_shed_mw", "max"))
                     .reset_index())
        yearly["shed_share_pct"] = 100 * yearly["total_shed_mwh"] / yearly["total_demand_mwh"]
        yearly.to_csv(out_dir / "per_region" / f"{region}_yearly{suffix}.csv", index=False)
    return out


def _yearly_national(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["year"] = pd.to_datetime(df["date"]).dt.year
    rows = []
    for year, sub in df.groupby("year"):
        by_reg = sub.groupby("region", observed=True)[["allocated_mwh", "demand_mwh"]].sum()
        rows.append({"year": year,
                     "demand_twh": sub["demand_mwh"].sum() / 1e6,
                     "shed_gwh": sub["shed_mwh"].sum() / 1e3,
                     "shed_share": sub["shed_mwh"].sum() / max(sub["demand_mwh"].sum(), 1e-6),
                     "shed_days": int((sub.groupby("date")["shed_mwh"].sum() > 1e-3).sum()),
                     "jain": jain_fairness((by_reg["allocated_mwh"] / by_reg["demand_mwh"]).values)})
    return pd.DataFrame(rows)


def write_markdown_summary(reports: Dict[str, pd.DataFrame], out_dir, assumptions,
                           policy_name: str, test_table: pd.DataFrame | None = None):
    out_dir = Path(out_dir)
    central = reports.get("central", next(iter(reports.values())))
    lines = ["# Load-Shedding Schedule Summary (2025 - 2040)", "",
             f"*Scheduling policy:* **{policy_name}**. Demand is a **scenario projection** "
             f"(not a validated forecast); capacity follows an explicit supply scenario.", "",
             "## Assumptions", "",
             f"* Demand CAGR low / central / high: {assumptions.cagr_low:.1%} / "
             f"{assumptions.cagr_central:.1%} / {assumptions.cagr_high:.1%} "
             f"(regional multipliers applied).",
             f"* Capacity: {assumptions.supply_base_mw:,.0f} MW in the base year "
             f"(P{int(CFG.grid.supply_base_quantile*100)} of 2024 served peak), growing "
             f"{assumptions.supply_cagr:.1%}/yr.",
             f"* Seasonality and weekend factor ({assumptions.weekend_discount:.3f}) calibrated "
             f"from 2022-2024 data; predictive band from measured forecaster MAPE "
             f"({assumptions.lstm_mape:.1%}).",
             "* Hourly shape is an assumed BD load curve normalised to the evening peak "
             "(the dataset is daily).", ""]
    if test_table is not None:
        lines += ["## Back-test on real 2024 days (hold-out)", "",
                  "| Policy | ENS (GWh) | ENS % | Peak shed MW/day | Priority-weighted MWh/day | Jain |",
                  "|:--|--:|--:|--:|--:|--:|"]
        for _, r in test_table.iterrows():
            lines.append(f"| {r['policy']} | {r['ens_total_gwh']:.2f} | {r['ens_pct_mean']:.3f} | "
                         f"{r['peak_shed_mw_mean']:.1f} | {r['priority_weighted_mwh_mean']:.1f} | "
                         f"{r['jain_served_ratio_mean']:.4f} |")
        lines.append("")
    lines += ["## Scenario comparison (cumulative 2025-2040)", "",
              "| Scenario | Demand (TWh) | Shed (GWh) | Shed share | Shed days |",
              "|:--|--:|--:|--:|--:|"]
    for name, rep in reports.items():
        y = _yearly_national(rep)
        lines.append(f"| {name} | {y['demand_twh'].sum():.1f} | {y['shed_gwh'].sum():.1f} | "
                     f"{y['shed_gwh'].sum() / max(y['demand_twh'].sum() * 1e3, 1e-9):.2%} | "
                     f"{int(y['shed_days'].sum())} |")
    lines += ["", "## Central scenario -- national totals by year", "",
              "| Year | Total demand (TWh) | Total shed (GWh) | Shed share | Shed days | Jain (served ratio) |",
              "|-----:|-------------------:|-----------------:|-----------:|----------:|--------------:|"]
    for _, r in _yearly_national(central).iterrows():
        lines.append(f"| {int(r['year'])} | {r['demand_twh']:.2f} | {r['shed_gwh']:.1f} | "
                     f"{r['shed_share']:.2%} | {int(r['shed_days'])} | {r['jain']:.4f} |")
    lines += ["", "## Central scenario -- per-region cumulative shed (2025 - 2040)", "",
              "| Region | Cumulative shed (GWh) | Shed share | Avg shed hours on shed days | Typical window | Priority |",
              "|:-------|----------------------:|-----------:|----------------------------:|:---------------|--------:|"]
    for region in REGIONS:
        sub = central[central["region"] == region]
        cum = sub["shed_mwh"].sum() / 1e3
        share = sub["shed_mwh"].sum() / max(sub["demand_mwh"].sum(), 1e-6)
        sd = sub[sub["shed_hours_count"] > 0]
        avg_h = sd["shed_hours_count"].mean() if len(sd) else 0.0
        win = sd["shed_hours"].mode().iat[0] if len(sd) else "-"
        lines.append(f"| {region} | {cum:.1f} | {share:.2%} | {avg_h:.2f} | {win} | "
                     f"{PRIORITY_SCORES_EXPERT[region]:.2f} |")
    lines += ["", "## Notes on interpretation (reviewer-compliant)", "",
              "* All shedding shown is **planned interruption** driven by the projected "
              "supply-demand balance, not unplanned outages (Comment 11: LOLP / LOLE excluded).",
              "* Shedding windows are **learned** by the MARL policy; no time slot is hard-coded.",
              "* `reason_code = allocation-limited` marks shedding that happened although national "
              "capacity was sufficient in that hour (a policy inefficiency, reported transparently).",
              "* Demand projections are **scenario-based** (Comment 13); low / central / high "
              "schedules are in the accompanying CSVs."]
    (out_dir / "daily_schedule_summary.md").write_text("\n".join(lines))

## 16. End-to-end pipeline

In [40]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

OUT = Path(OUTPUT_DIR)


def build_graph_tensors(device=DEVICE):
    """Adjacency and edge attributes in REGIONS order."""
    idx = {r: i for i, r in enumerate(REGIONS)}
    N = len(REGIONS)
    adj = torch.zeros(N, N)
    edge_attr = torch.zeros(N, N, 2)
    for u, v, a in GRID_EDGES:
        i, j = idx[u], idx[v]
        adj[i, j] = adj[j, i] = 1.0
        edge_attr[i, j] = edge_attr[j, i] = torch.tensor(
            [a["capacity_mw"] / 1000.0, a["kv"] / 400.0])
    return adj.to(device), edge_attr.to(device)


def conformal_halfwidth(y_val_mw, p_val_mw, alpha=0.05):
    """Split-conformal |residual| quantile per (region, horizon)."""
    n = y_val_mw.shape[0]
    q = min(1.0, np.ceil((n + 1) * (1 - alpha)) / n)
    return np.quantile(np.abs(y_val_mw - p_val_mw), q, axis=0)       # (N, H)


def evaluate_forecaster_with_uncertainty(model, X_test, y_test, adj, edge_attr,
                                         scaler, X_val=None, y_val=None,
                                         n_samples=50, alpha=0.05):
    """All metrics in MW. y_* are SCALED targets (as produced by windows_for)."""
    mean_t, lo_t, hi_t = model.predict_with_uncertainty(
        torch.from_numpy(X_test), adj, edge_attr, n_samples=n_samples, alpha=alpha)
    y = scaler.inverse_y(y_test)
    p = scaler.inverse_y(mean_t.numpy())
    l = scaler.inverse_y(lo_t.numpy()); u = scaler.inverse_y(hi_t.numpy())
    out = {**point_metrics(y, p),
           "PICP_95_mc": picp(y, l, u), "MPIW_mc": mpiw(l, u),
           "Winkler_05_mc": winkler_score(y, l, u, alpha=alpha),
           "n_test": int(np.prod(y.shape))}
    arrays = {"y": y, "p": p, "lo": l, "hi": u}
    if X_val is not None:
        mv, _, _ = model.predict_with_uncertainty(torch.from_numpy(X_val), adj, edge_attr,
                                                  n_samples=n_samples, alpha=alpha)
        q = conformal_halfwidth(scaler.inverse_y(y_val), scaler.inverse_y(mv.numpy()), alpha)
        lc, uc = p - q[None], p + q[None]
        out.update({"PICP_95_conformal": picp(y, lc, uc), "MPIW_conformal": mpiw(lc, uc),
                    "Winkler_05_conformal": winkler_score(y, lc, uc, alpha=alpha)})
        arrays.update({"lo_conf": lc, "hi_conf": uc})
    return out, arrays


def build_env_samples(model, wd, split, scaler, adj, edge_attr, realized_pv,
                      extra_noise_rel=None, seed=0):
    """
    One EnvSample per REAL target day of the split (the first day of each window):
      demand_forecast = GNN-LSTM day-ahead forecast (MW) x hourly profile
      demand_actual   = realised demand (MW) x hourly profile
      supply_cap_mw   = realised national served peak on stress days, and
                        served peak x (1 + reserve margin) otherwise
      gcn_embedding   = learned GCN embedding of the issue day
    extra_noise_rel (N,) optionally inflates in-sample forecast errors on the
    TRAINING days up to the validation error level.
    """
    rng = np.random.default_rng(seed)
    o = wd["splits"][split]
    X, _ = windows_for(wd, split, scaler)
    fc = scaler.inverse_y(predict_point(model, X, adj, edge_attr))[:, :, 0]    # (S, N)
    if extra_noise_rel is not None:
        fc = fc * (1.0 + rng.normal(0.0, 1.0, fc.shape) * extra_noise_rel[None, :])
    emb = embed_windows(model, X, adj, edge_attr)                              # (S, N, E)
    rm_idx = wd["feature_cols"].index("rolling_mean_7d")
    trailing = wd["F"][o, :, rm_idx]                     # mean of days t-7..t-1 (MW)
    tdates = wd["dates"][o + 1]
    actual = wd["Y"][o + 1]
    sup = realized_pv["supply"].reindex(tdates).values
    shed = realized_pv["shed"].reindex(tdates).values
    samples = []
    for k in range(len(o)):
        stress = shed[k].sum() > CFG.grid.stress_threshold_mw
        cap = sup[k].sum() * (1.0 if stress else 1.0 + CFG.grid.unconstrained_reserve_margin)
        samples.append(EnvSample(
            date=tdates[k],
            demand_forecast=np.maximum(fc[k], 0)[:, None] * HOURLY_PROFILE[None, :],
            demand_actual=actual[k][:, None] * HOURLY_PROFILE[None, :],
            supply_cap_mw=float(cap), gcn_embedding=emb[k].astype(np.float32),
            trailing_mean=trailing[k].astype(np.float64),
            supply_realized=sup[k].astype(np.float64), is_stress=bool(stress)))
    return samples


def baseline_granted(samples, name, prio=PRIO_VEC, hist_share=None):
    """Granted MW (D, N, 24) of a scheduling baseline over a list of days."""
    out = []
    for s in samples:
        if name == "HistoricalRealized":
            g = historical_realized(s.demand_actual, s.supply_realized)
        elif name == "HistoricalShare":
            g = historical_share(s.demand_forecast, s.supply_cap_mw, hist_share)
        elif name == "ProportionalToForecast":
            g = proportional_to_forecast(s.demand_forecast, s.supply_cap_mw)
        elif name == "PriorityHeuristic":
            g = priority_heuristic(s.demand_forecast, s.supply_cap_mw, prio)
        elif name == "GreedySmallestFirst":
            g = greedy_smallest_first(s.demand_forecast, s.supply_cap_mw)
        elif name == "LPOracle_perfect_foresight":
            g = milp_optimum(s.demand_actual, s.supply_cap_mw, prio)
        else:
            raise KeyError(name)
        out.append(g)
    return np.stack(out)


def forecast_error_by_region(model, wd, split, scaler, adj, edge_attr):
    X, y = windows_for(wd, split, scaler)
    p = scaler.inverse_y(predict_point(model, X, adj, edge_attr))[:, :, 0]
    yy = scaler.inverse_y(y)[:, :, 0]
    return np.sqrt(np.mean(((p - yy) / yy) ** 2, axis=0))            # relative RMSE (N,)

### 16.1  Load data + leakage-safe features + windows

In [41]:
raw   = load_raw(RAW_XLSX)                       # wide, daily, continuous calendar
feats = build_leakage_safe_features(raw)         # long, leakage-safe
realized = extract_realized(raw)                 # realised supply / shed (NOT features)
realized_pv = {
    "supply": realized.pivot(index="timestamp", columns="region",
                             values="supply_mw_realized")[REGIONS],
    "shed": realized.pivot(index="timestamp", columns="region",
                           values="load_shed_mw_realized")[REGIONS],
}

wd = build_window_dataset(feats)
scaler = PanelScaler().fit(wd, CFG.data.train_end_date)
X_train, y_train = windows_for(wd, "train", scaler)
X_val,   y_val   = windows_for(wd, "val",   scaler)
X_test,  y_test  = windows_for(wd, "test",  scaler)

adj, edge_attr = build_graph_tensors()
n_features = X_train.shape[-1]
L, H = CFG.data.lookback_days, CFG.data.horizon_days

assert not set(wd["feature_cols"]) & set(CFG.data.forbidden_leaky), "leakage!"
print("features   :", wd["feature_cols"])
print("X_train:", X_train.shape, " X_val:", X_val.shape, " X_test:", X_test.shape,
      " (samples, lookback, regions, features)")
print("node order :", REGIONS)
for k in ("train", "val", "test"):
    o = wd["splits"][k]
    print(f"{k:5s} targets: {wd['dates'][o[0]+1].date()} -> {wd['dates'][o[-1]+H].date()}"
          f"  ({len(o)} windows)")
nat = realized.groupby("timestamp")[["demand_mw", "load_shed_mw_realized"]].sum()
print("\nshare of days with recorded shedding per year:")
print((nat["load_shed_mw_realized"] > 0.5).groupby(nat.index.year).mean().round(2).to_dict())

load_raw: 1850 raw rows -> 1867 calendar days (17 gap days interpolated)
features   : ['day_of_week_num', 'month', 'year', 'is_weekend', 'doy_sin', 'doy_cos', 'temperature_forecast', 'holiday_cat', 'demand_obs', 'demand_lag_1d', 'demand_lag_7d', 'demand_lag_30d', 'rolling_mean_7d', 'rolling_std_7d']
X_train: (1033, 60, 9, 14)  X_val: (317, 60, 9, 14)  X_test: (332, 60, 9, 14)  (samples, lookback, regions, features)
node order : ['Dhaka', 'Chattogram', 'Khulna', 'Rajshahi', 'Sylhet', 'Barishal', 'Rangpur', 'Mymensingh', 'Cumilla']
train targets: 2020-02-19 -> 2022-12-31  (1033 windows)
val   targets: 2023-01-01 -> 2023-12-31  (317 windows)
test  targets: 2024-01-01 -> 2024-12-29  (332 windows)

share of days with recorded shedding per year:
{2019: 0.0, 2020: 0.0, 2021: 0.0, 2022: 0.32, 2023: 0.62, 2024: 0.61}


### 16.2  Optuna-tune the forecaster

In [42]:
B = CFG.budget
t0 = time.time()
fc_objective = build_forecaster_objective(
    X_train, y_train, X_val, y_val, adj, edge_attr, scaler,
    n_features, len(REGIONS), L, H,
    max_epochs=B.fc_tune_epochs, patience=max(3, B.fc_patience // 2))
study = run_forecaster_tuning(fc_objective, n_trials=B.fc_trials)
best_fc = study.best_params
Path(f"{OUTPUT_DIR}/optuna_forecaster_best.json").write_text(
    json.dumps({**best_fc, "best_val_rmse_mw": study.best_value}, indent=2))
study.trials_dataframe().to_csv(f"{OUTPUT_DIR}/optuna_forecaster_trials.csv", index=False)
print(f"Best forecaster HPs: {best_fc}")
print(f"Best validation RMSE: {study.best_value:.1f} MW   ({time.time()-t0:.0f}s)")

  0%|          | 0/12 [00:00<?, ?it/s]

Best forecaster HPs: {'gcn_hidden': 128, 'lstm_hidden': 256, 'dropout': 0.1422772674924288, 'lr': 0.0015308837415731389, 'batch_size': 64, 'weight_decay': 3.945908811099999e-06}
Best validation RMSE: 172.4 MW   (136s)


### 16.3  Train final forecaster, evaluate, compare with forecasting baselines

In [43]:
def make_forecaster(params, use_graph=True, train_gcn=True, seed=42):
    seed_everything(seed)
    m = GNNLSTMForecaster(n_features, len(REGIONS), L, H,
                          gcn_hidden=params["gcn_hidden"],
                          lstm_hidden=params["lstm_hidden"],
                          dropout=params["dropout"],
                          residual_idx=scaler.residual_idx).to(DEVICE)
    if not train_gcn:
        m.freeze_gcn()
    return m


t0 = time.time()
model = make_forecaster(best_fc)
model, best_val, fc_hist = fit_forecaster(
    model, X_train, y_train, X_val, y_val, adj, edge_attr, scaler,
    lr=best_fc["lr"], batch_size=best_fc["batch_size"],
    epochs=CFG.budget.fc_final_epochs, patience=CFG.budget.fc_patience,
    weight_decay=best_fc.get("weight_decay", 1e-5), verbose=True)
torch.save(model.state_dict(), f"{OUTPUT_DIR}/gnn_lstm_forecaster.pt")
pd.DataFrame(fc_hist).to_csv(f"{OUTPUT_DIR}/forecaster_training_history.csv", index=False)
print(f"trained in {time.time()-t0:.0f}s, best val RMSE {best_val:.1f} MW")

fc_metrics, fc_arrays = evaluate_forecaster_with_uncertainty(
    model, X_test, y_test, adj, edge_attr, scaler, X_val, y_val,
    n_samples=CFG.budget.mc_samples)
pd.DataFrame([fc_metrics]).to_csv(f"{OUTPUT_DIR}/forecasting_metrics.csv", index=False)

# ---- Forecasting baselines on the SAME test windows (Reviewer Comment 17) ----
y_test_mw = scaler.inverse_y(y_test)
fc_table = []
for name, pred in naive_baselines_from_windows(wd, "test").items():
    fc_table.append({"model": name, **point_metrics(y_test_mw, pred)})
Xtr_tab, ytr_tab = tabular_from_windows(X_train, y_train)
Xte_tab, _ = tabular_from_windows(X_test)
for name, mdl in [("Ridge", RidgeBaseline(H, alpha=1.0)), ("XGBoost" if _HAS_XGB else "RandomForest", TreeBaseline(H))]:
    mdl.fit(Xtr_tab, ytr_tab)
    pred = scaler.inverse_y(mdl.predict(Xte_tab).reshape(y_test.shape))
    fc_table.append({"model": name, **point_metrics(y_test_mw, pred)})
fc_table.append({"model": "GNN-LSTM (MC mean)", **point_metrics(y_test_mw, fc_arrays["p"])})
fc_table = pd.DataFrame(fc_table)
fc_table.to_csv(f"{OUTPUT_DIR}/forecasting_baselines_comparison.csv", index=False)

per_region = pd.DataFrame([{"region": r, **point_metrics(y_test_mw[:, i], fc_arrays["p"][:, i])}
                           for i, r in enumerate(REGIONS)])
per_region.to_csv(f"{OUTPUT_DIR}/forecasting_metrics_per_region.csv", index=False)
per_h = pd.DataFrame([{"horizon_day": h + 1, **point_metrics(y_test_mw[:, :, h], fc_arrays["p"][:, :, h])}
                      for h in range(H)])
per_h.to_csv(f"{OUTPUT_DIR}/forecasting_metrics_per_horizon.csv", index=False)

print("\nTest-set forecasting comparison (MW, 2024 hold-out):")
display(fc_table.round(3))
print("GNN-LSTM interval metrics:")
display(pd.DataFrame([{k: v for k, v in fc_metrics.items() if "PICP" in k or "MPIW" in k or "Winkler" in k}]).round(3))
display(per_region.round(3))

epoch   0  train_mse(scaled) 0.3203  val RMSE   200.67 MW
epoch   5  train_mse(scaled) 0.2094  val RMSE   182.60 MW
epoch  10  train_mse(scaled) 0.1684  val RMSE   182.56 MW
trained in 24s, best val RMSE 174.5 MW

Test-set forecasting comparison (MW, 2024 hold-out):


,model,MAE,RMSE,MAPE,sMAPE,R2,CV_RMSE_pct
0,Persistence,122.929,234.134,9.069,8.794,0.961,16.964
1,SeasonalNaive_7d,138.004,268.611,10.237,9.914,0.949,19.462
2,Ridge,106.603,199.445,8.258,7.769,0.972,14.451
3,XGBoost,124.645,223.608,8.715,8.941,0.965,16.202
4,GNN-LSTM (MC mean),104.428,199.415,7.913,7.441,0.972,14.449


GNN-LSTM interval metrics:


,PICP_95_mc,MPIW_mc,Winkler_05_mc,PICP_95_conformal,MPIW_conformal,Winkler_05_conformal
0,0.203,52.598,3288.62,0.922,525.383,886.809


,region,MAE,RMSE,MAPE,sMAPE,R2,CV_RMSE_pct
0,Dhaka,360.086,513.745,8.655,8.155,0.488,11.443
1,Chattogram,89.495,119.605,7.672,7.337,0.507,9.924
2,Khulna,101.446,143.507,6.810,6.554,0.753,9.515
3,Rajshahi,95.389,135.575,7.452,7.137,0.738,10.458
4,Sylhet,38.540,56.773,8.114,7.629,0.722,11.615
5,Barishal,33.068,51.988,9.860,8.625,0.632,13.598
6,Rangpur,62.879,91.036,8.010,7.541,0.569,10.900
7,Mymensingh,75.523,107.922,7.284,7.018,0.604,10.207
8,Cumilla,83.427,121.739,7.362,6.973,0.740,10.511


### 16.4  Build the real environment, Optuna-tune MAPPO, train the final policy, compare with scheduling baselines

In [44]:
# ---------------- real environment from the trained forecaster ----------------
rel_tr = forecast_error_by_region(model, wd, "train", scaler, adj, edge_attr)
rel_va = forecast_error_by_region(model, wd, "val", scaler, adj, edge_attr)
extra_noise = np.sqrt(np.maximum(rel_va ** 2 - rel_tr ** 2, 0.0))
print("in-sample vs validation relative RMSE per region:")
print(pd.DataFrame({"train": rel_tr, "val": rel_va, "added_noise": extra_noise}, index=REGIONS).round(4))


def make_env_sets(fmodel, seed=0):
    return (build_env_samples(fmodel, wd, "train", scaler, adj, edge_attr, realized_pv,
                              extra_noise_rel=extra_noise, seed=seed),
            build_env_samples(fmodel, wd, "val", scaler, adj, edge_attr, realized_pv),
            build_env_samples(fmodel, wd, "test", scaler, adj, edge_attr, realized_pv))


train_samples, val_samples, test_samples = make_env_sets(model)
print(f"env days  train {len(train_samples)} ({sum(s.is_stress for s in train_samples)} stress)"
      f" | val {len(val_samples)} ({sum(s.is_stress for s in val_samples)} stress)"
      f" | test {len(test_samples)} ({sum(s.is_stress for s in test_samples)} stress)")

emb_train = np.stack([s.gcn_embedding for s in train_samples])
fit_obs_embedding_projection(emb_train, CFG.marl.obs_embed_dim)
CFG.marl.obs_dim_per_agent = obs_dim_for(emb_train.shape[-1])
base_marl = CFG.marl
train_env_factory = lambda seed: LoadShedEnv(train_samples, seed=seed,
                                             episode_len=base_marl.episode_len)
val_env = LoadShedEnv(val_samples, episode_len=None, random_start=False)
test_env = LoadShedEnv(test_samples, episode_len=None, random_start=False)

# ---------------- Optuna ----------------
t0 = time.time()
m_obj = build_marl_objective(train_env_factory, val_env, base_marl,
                             seeds=base_marl.seeds[:CFG.budget.marl_tune_seeds],
                             train_steps=CFG.budget.marl_tune_steps, emb_train=emb_train)
m_study = run_marl_tuning(m_obj, n_trials=CFG.budget.marl_trials)
best_marl = m_study.best_params
Path(f"{OUTPUT_DIR}/optuna_marl_best.json").write_text(
    json.dumps({**best_marl, "best_val_reward": m_study.best_value}, indent=2))
m_study.trials_dataframe().to_csv(f"{OUTPUT_DIR}/optuna_marl_trials.csv", index=False)
print(f"Best MAPPO HPs: {best_marl}  (val reward {m_study.best_value:.4f}, {time.time()-t0:.0f}s)")

# ---------------- final MAPPO policy ----------------
k_best = best_marl.get("obs_embed_dim", base_marl.obs_embed_dim)
expl = fit_obs_embedding_projection(emb_train, k_best)
MAIN_PROJ = {k: (None if v is None else v.copy()) for k, v in OBS_EMBED_PROJ.items()}
MARL_BEST = marl_cfg_from_params(base_marl, best_marl,
                                 total_env_steps=CFG.budget.marl_final_steps,
                                 obs_dim_per_agent=obs_dim_for(emb_train.shape[-1]))
print(f"agents observe {k_best} PCA dims of the GCN embedding (explained variance {expl:.2f})")
seed_everything(42)
mappo = MAPPO(MARL_BEST)
t0 = time.time()
train_log = train_mappo(mappo, train_env_factory(42), MARL_BEST.total_env_steps,
                        eval_fn=lambda m: evaluate_policy(m, val_env),
                        eval_every_updates=max(1, MARL_BEST.total_env_steps // MARL_BEST.rollout_len // 20),
                        verbose=True)
torch.save(mappo.state_dict(), f"{OUTPUT_DIR}/mappo_policy.pt")
pd.DataFrame(train_log["log"]).to_csv(f"{OUTPUT_DIR}/mappo_training_log.csv", index=False)
print(f"MAPPO trained in {time.time()-t0:.0f}s; best val reward {train_log['best_eval']:.4f}")

# ---------------- scheduling comparison on the 2024 hold-out ----------------
Y_tr = wd["Y"][wd["dates"] <= pd.Timestamp(CFG.data.train_end_date)]
HIST_SHARE = Y_tr.mean(0) / Y_tr.mean(0).sum()
POLICIES = ["HistoricalRealized", "HistoricalShare", "ProportionalToForecast",
            "PriorityHeuristic", "GreedySmallestFirst", "LPOracle_perfect_foresight"]


def compare_policies(samples, mappo_model, prio=PRIO_VEC, label="test"):
    env = LoadShedEnv(samples, episode_len=None, random_start=False, prio=prio)
    tables = {"MAPPO (GNN-LSTM obs)": schedule_day_table(
        samples, evaluate_on_env(mappo_model, env)["granted"], prio)}
    for name in POLICIES:
        tables[name] = schedule_day_table(samples, baseline_granted(samples, name, prio, HIST_SHARE), prio)
    summary = pd.DataFrame([summarize_schedule(t, n) for n, t in tables.items()])
    return summary, tables


sched_val_summary, _ = compare_policies(val_samples, mappo)          # for policy selection
sched_val_summary.to_csv(f"{OUTPUT_DIR}/scheduling_comparison_val.csv", index=False)
sched_summary, sched_tables = compare_policies(test_samples, mappo)
sched_summary.to_csv(f"{OUTPUT_DIR}/scheduling_comparison_test.csv", index=False)
mp = sched_tables["MAPPO (GNN-LSTM obs)"]
paired = []
for name in POLICIES:
    for col in ["ens_mwh", "priority_weighted_mwh", "peak_shed_mw"]:
        paired.append({"baseline": name, "metric": col,
                       **paired_bootstrap_diff(mp[col].values, sched_tables[name][col].values)})
paired = pd.DataFrame(paired)
paired.to_csv(f"{OUTPUT_DIR}/scheduling_paired_tests_test.csv", index=False)
pd.concat([t.assign(policy=n) for n, t in sched_tables.items()]).to_csv(
    f"{OUTPUT_DIR}/scheduling_daily_test.csv", index=False)

cols = ["policy", "ens_total_gwh", "ens_pct_mean", "peak_shed_mw_mean",
        "priority_weighted_mwh_mean", "jain_served_ratio_mean",
        "worst_region_shed_pct_mean", "over_dispatch_pct_mean", "gini_region_shed_total"]
print("Scheduling on real 2024 days (MAPPO minus baseline: negative = MAPPO better):")
display(sched_summary[cols].round(4))
display(paired.round(3))

in-sample vs validation relative RMSE per region:
             train     val  added_noise
Dhaka       0.0789  0.0889       0.0410
Chattogram  0.0759  0.0699       0.0000
Khulna      0.0625  0.0737       0.0391
Rajshahi    0.0652  0.0675       0.0177
Sylhet      0.0756  0.0739       0.0000
Barishal    0.0717  0.0685       0.0000
Rangpur     0.0737  0.0751       0.0141
Mymensingh  0.0681  0.0694       0.0131
Cumilla     0.0644  0.0668       0.0174
env days  train 1033 (118 stress) | val 317 (198 stress) | test 332 (200 stress)


  0%|          | 0/8 [00:00<?, ?it/s]

Best MAPPO HPs: {'hidden_dim': 64, 'lr_actor': 1.3400367243354803e-05, 'lr_critic': 0.0002879775265707035, 'gamma': 0.30892415592540967, 'gae_lambda': 0.6931258694211608, 'clip_ratio': 0.2275114942710426, 'entropy_coef': 0.0045881565491609705, 'team_reward_weight': 0.7361074625809747, 'n_action_basis': 8, 'obs_embed_dim': 2}  (val reward -1.5927, 1385s)
agents observe 2 PCA dims of the GCN embedding (explained variance 0.69)
  update   11 step    2816  eval   -1.5866  entropy -0.65
  update   22 step    5632  eval   -1.6298  entropy -0.66
  update   33 step    8448  eval   -1.6744  entropy -0.66
  update   44 step   11264  eval   -1.7156  entropy -0.66
  update   55 step   14080  eval   -1.7579  entropy -0.66
  update   66 step   16896  eval   -1.7997  entropy -0.67
  update   77 step   19712  eval   -1.8233  entropy -0.67
  update   88 step   22528  eval   -1.8410  entropy -0.67
  update   99 step   25344  eval   -1.8594  entropy -0.67
  update  110 step   28160  eval   -1.8637  entro

,policy,ens_total_gwh,ens_pct_mean,peak_shed_mw_mean,priority_weighted_mwh_mean,jain_served_ratio_mean,worst_region_shed_pct_mean,over_dispatch_pct_mean,gini_region_shed_total
0,MAPPO (GNN-LSTM obs),658.0768,0.7187,323.1454,1449.3315,0.9998,2.8191,2.6156,0.2740
1,HistoricalRealized,514.0514,0.5381,221.2229,991.7582,0.9998,2.7547,0.0000,0.4604
2,HistoricalShare,1112.5716,1.2713,485.1841,2383.6526,0.9995,5.4622,21.4769,0.4057
3,ProportionalToForecast,627.5248,0.6828,323.6331,1405.7666,0.9999,2.4470,20.8884,0.3111
4,PriorityHeuristic,976.0065,1.1248,486.9874,1689.0351,0.9930,15.4917,21.3305,0.5424
5,GreedySmallestFirst,713.6983,0.7810,384.4932,1822.8094,0.9999,2.4985,20.9866,0.5968
6,LPOracle_perfect_foresight,327.0832,0.3360,217.0396,457.8325,0.9972,7.6169,0.0000,0.8235


,baseline,metric,mean_diff,ci95_lo,ci95_hi,wilcoxon_p
0,HistoricalRealized,ens_mwh,433.811,300.903,572.948,0.000
1,HistoricalRealized,priority_weighted_mwh,457.573,358.367,565.588,0.000
2,HistoricalRealized,peak_shed_mw,101.922,88.839,115.327,0.000
3,HistoricalShare,ens_mwh,-1368.960,-1536.972,-1203.170,0.000
4,HistoricalShare,priority_weighted_mwh,-934.321,-1064.758,-802.748,0.000
5,HistoricalShare,peak_shed_mw,-162.039,-178.399,-145.564,0.000
6,ProportionalToForecast,ens_mwh,92.024,54.285,135.910,0.000
7,ProportionalToForecast,priority_weighted_mwh,43.565,13.630,80.062,0.932
8,ProportionalToForecast,peak_shed_mw,-0.488,-1.265,0.292,0.028
9,PriorityHeuristic,ens_mwh,-957.620,-1196.775,-738.684,0.000


### 16.5  Ablation grid across seeds (real training runs)

In [45]:
id_adj, id_edge = build_identity_graph(len(REGIONS), DEVICE)
_ablation_fc_cache = {}


def build_and_evaluate(cfg_dict, seed):
    """Train the forecaster variant (cached per graph setting and seed), then
    schedule the 2024 hold-out with MAPPO (use_rl) or ProportionalToForecast."""
    key = (cfg_dict["use_graph"], cfg_dict["train_gcn"], seed)
    a, e = (adj, edge_attr) if cfg_dict["use_graph"] else (id_adj, id_edge)
    if key not in _ablation_fc_cache:
        fm = make_forecaster(best_fc, cfg_dict["use_graph"], cfg_dict["train_gcn"], seed)
        fm, _, _ = fit_forecaster(fm, X_train, y_train, X_val, y_val, a, e, scaler,
                                  lr=best_fc["lr"], batch_size=best_fc["batch_size"],
                                  epochs=CFG.budget.ablation_fc_epochs,
                                  patience=CFG.budget.fc_patience,
                                  weight_decay=best_fc.get("weight_decay", 1e-5), seed=seed)
        p = scaler.inverse_y(predict_point(fm, X_test, a, e))
        _ablation_fc_cache[key] = (fm, point_metrics(y_test_mw, p))
    fm, fcm = _ablation_fc_cache[key]

    # env built with THIS variant's forecasts and embeddings
    tr_s = build_env_samples(fm, wd, "train", scaler, a, e, realized_pv, extra_noise, seed)
    va_s = build_env_samples(fm, wd, "val", scaler, a, e, realized_pv)
    te_s = build_env_samples(fm, wd, "test", scaler, a, e, realized_pv)
    if cfg_dict["use_rl"]:
        seed_everything(seed)
        # this variant's own embedding PCA (same k as the tuned policy)
        fit_obs_embedding_projection(np.stack([s.gcn_embedding for s in tr_s]),
                                     MARL_BEST.obs_embed_dim)
        mcfg = marl_cfg_from_params(MARL_BEST, {}, total_env_steps=CFG.budget.ablation_marl_steps)
        agent = MAPPO(mcfg)
        v_env = LoadShedEnv(va_s, episode_len=None, random_start=False)
        train_mappo(agent, LoadShedEnv(tr_s, seed=seed, episode_len=mcfg.episode_len),
                    mcfg.total_env_steps, eval_fn=lambda m: evaluate_policy(m, v_env),
                    eval_every_updates=max(1, mcfg.total_env_steps // mcfg.rollout_len // 5))
        granted = evaluate_on_env(agent, LoadShedEnv(te_s, episode_len=None,
                                                     random_start=False))["granted"]
        OBS_EMBED_PROJ.update(MAIN_PROJ)                  # restore main projection
    else:
        granted = baseline_granted(te_s, "ProportionalToForecast")
    s = summarize_schedule(schedule_day_table(te_s, granted, PRIO_VEC), cfg_dict["name"])
    return {"fc_RMSE_MW": fcm["RMSE"], "fc_MAPE": fcm["MAPE"],
            "ens_gwh": s["ens_total_gwh"], "ens_pct": s["ens_pct_mean"],
            "priority_weighted_mwh_day": s["priority_weighted_mwh_mean"],
            "jain": s["jain_served_ratio_mean"]}


t0 = time.time()
ablation_df, ablation_raw = run_ablation(build_and_evaluate, seeds=CFG.budget.ablation_seeds)
ablation_df.to_csv(f"{OUTPUT_DIR}/ablation_results.csv", index=False)
ablation_raw.to_csv(f"{OUTPUT_DIR}/ablation_results_per_seed.csv", index=False)
print(f"ablation finished in {time.time()-t0:.0f}s")
display(ablation_df[["config", "n_seeds"] + [c for c in ablation_df.columns if c.endswith("_mean")]].round(4))

  LSTM_only              seed 7    fc_RMSE_MW=189.802  fc_MAPE=7.685  ens_gwh=633.691  ens_pct=0.688  priority_weighted_mwh_day=1408.812  jain=1.000   (33s)
  LSTM_only              seed 13   fc_RMSE_MW=193.979  fc_MAPE=8.192  ens_gwh=641.227  ens_pct=0.697  priority_weighted_mwh_day=1438.987  jain=1.000   (29s)
  LSTM_only              seed 29   fc_RMSE_MW=190.668  fc_MAPE=7.947  ens_gwh=639.040  ens_pct=0.694  priority_weighted_mwh_day=1423.431  jain=1.000   (27s)
  GNN_LSTM_trained       seed 7    fc_RMSE_MW=198.802  fc_MAPE=8.000  ens_gwh=625.794  ens_pct=0.680  priority_weighted_mwh_day=1396.198  jain=1.000   (33s)
  GNN_LSTM_trained       seed 13   fc_RMSE_MW=199.711  fc_MAPE=8.200  ens_gwh=632.706  ens_pct=0.688  priority_weighted_mwh_day=1419.846  jain=1.000   (26s)
  GNN_LSTM_trained       seed 29   fc_RMSE_MW=199.462  fc_MAPE=8.323  ens_gwh=644.219  ens_pct=0.701  priority_weighted_mwh_day=1438.241  jain=1.000   (26s)
  GNN_LSTM_random_GCN    seed 7    fc_RMSE_MW=212.345  fc_

,config,n_seeds,fc_RMSE_MW_mean,fc_MAPE_mean,ens_gwh_mean,ens_pct_mean,priority_weighted_mwh_day_mean,jain_mean
0,LSTM_only,3,191.4829,7.9412,637.9861,0.6930,1423.7436,0.9999
1,GNN_LSTM_trained,3,199.3247,8.1746,634.2398,0.6899,1418.0948,0.9999
2,GNN_LSTM_random_GCN,3,208.0973,8.4320,683.3746,0.7454,1523.1180,0.9998
3,LSTM_PPO,3,191.4829,7.9412,686.0382,0.7475,1503.9985,0.9998
4,GNN_LSTM_PPO_full,3,199.3247,8.1746,666.6221,0.7276,1464.8347,0.9998


### 16.6  Sensitivity: economic, priority scores, reward weights

In [46]:
# ---- Economic sensitivity with the MEASURED ENS reduction (MAPPO vs history) ----
hist_row = sched_summary.set_index("policy").loc["HistoricalRealized"]
mappo_row = sched_summary.set_index("policy").loc["MAPPO (GNN-LSTM obs)"]
days = mappo_row["days"]
annual_red_mwh = (hist_row["ens_total_gwh"] - mappo_row["ens_total_gwh"]) * 1e3 * 365.0 / days
print(f"Measured ENS reduction vs historical dispatch: {annual_red_mwh:,.0f} MWh/yr "
      f"(test period, annualised)")
if annual_red_mwh <= 0:
    print("  -> MAPPO did not reduce ENS vs history on the hold-out; economics below "
          "will show negative NPV. (Increase RUN_PROFILE budget or revisit reward weights.)")
econ_df = economic_sweep(annual_ens_reduction_mwh=max(annual_red_mwh, 0.0),
                         evaluate_econ_fn=economic_metrics)
econ_df.to_csv(f"{OUTPUT_DIR}/sensitivity_economic.csv", index=False)
display(econ_df.head(10))

# ---- Priority-score sensitivity (policies re-evaluated under perturbed priorities) ----
prop_granted_test = baseline_granted(test_samples, "ProportionalToForecast")


def eval_priorities(p):
    """The MAPPO agents observe their own priority, so they are re-run with p."""
    out = {}
    g_mappo = evaluate_on_env(mappo, LoadShedEnv(test_samples, prio=p, episode_len=None,
                                                 random_start=False))["granted"]
    for name, g in [("MAPPO", g_mappo),
                    ("Priority", baseline_granted(test_samples, "PriorityHeuristic", p)),
                    ("Proportional", prop_granted_test)]:
        t = schedule_day_table(test_samples, g, p)
        out[f"{name}_prio_mwh_day"] = float(t["priority_weighted_mwh"].mean())
        out[f"{name}_ens_gwh"] = float(t["ens_mwh"].sum() / 1e3)
    out["MAPPO_beats_priority"] = out["MAPPO_prio_mwh_day"] <= out["Priority_prio_mwh_day"]
    return out


prio_df = priority_score_sweep(eval_priorities, n_random=CFG.budget.priority_mc_draws)
prio_df.to_csv(f"{OUTPUT_DIR}/sensitivity_priority.csv", index=False)
mc = prio_df[prio_df["kind"] == "per_region_mc"]
print(f"per-region MC draws: {len(mc)}, ranking changed in {mc['rank_changed'].mean():.0%}; "
      f"MAPPO <= PriorityHeuristic on priority-weighted burden in "
      f"{mc['MAPPO_beats_priority'].mean():.0%} of draws")
display(prio_df[prio_df["kind"] == "uniform"].round(3))

# ---- Reward-weight sensitivity (one-at-a-time, each point = a MAPPO training) ----
if CFG.budget.run_reward_sweep:
    def eval_reward_weights(rw):
        seed_everything(42)
        mcfg = marl_cfg_from_params(MARL_BEST, {}, total_env_steps=CFG.budget.reward_sweep_steps)
        agent = MAPPO(mcfg)
        v_env = LoadShedEnv(val_samples, reward_weights=rw, episode_len=None, random_start=False)
        train_mappo(agent, LoadShedEnv(train_samples, reward_weights=rw, seed=42,
                                       episode_len=mcfg.episode_len),
                    mcfg.total_env_steps, eval_fn=lambda m: evaluate_policy(m, v_env),
                    eval_every_updates=max(1, mcfg.total_env_steps // mcfg.rollout_len // 4))
        g = evaluate_on_env(agent, LoadShedEnv(test_samples, reward_weights=rw,
                                               episode_len=None, random_start=False))["granted"]
        s = summarize_schedule(schedule_day_table(test_samples, g, PRIO_VEC), "sweep")
        return {"ens_gwh": s["ens_total_gwh"], "priority_weighted_mwh_day": s["priority_weighted_mwh_mean"],
                "jain": s["jain_served_ratio_mean"], "over_dispatch_pct": s["over_dispatch_pct_mean"]}

    t0 = time.time()
    reward_df = reward_weight_sweep(eval_reward_weights, mode="oat")
    reward_df.to_csv(f"{OUTPUT_DIR}/sensitivity_reward_weights.csv", index=False)
    print(f"reward sweep: {len(reward_df)} trainings in {time.time()-t0:.0f}s")
    display(reward_df.round(4))

Measured ENS reduction vs historical dispatch: -158,341 MWh/yr (test period, annualised)
  -> MAPPO did not reduce ENS vs history on the hold-out; economics below will show negative NPV. (Increase RUN_PROFILE budget or revisit reward weights.)


,voll_bdt_per_kwh,impl_cost_bdt,discount_rate,ens_scale,NPV_BDT,BCR,payback_years
0,22.5,250000000.0,0.06,0.5,-6.180044e+08,0.0,inf
1,22.5,250000000.0,0.06,1.0,-6.180044e+08,0.0,inf
2,22.5,250000000.0,0.10,0.5,-5.572284e+08,0.0,inf
3,22.5,250000000.0,0.10,1.0,-5.572284e+08,0.0,inf
4,22.5,250000000.0,0.14,0.5,-5.108058e+08,0.0,inf
5,22.5,250000000.0,0.14,1.0,-5.108058e+08,0.0,inf
6,22.5,500000000.0,0.06,0.5,-8.680044e+08,0.0,inf
7,22.5,500000000.0,0.06,1.0,-8.680044e+08,0.0,inf
8,22.5,500000000.0,0.10,0.5,-8.072284e+08,0.0,inf
9,22.5,500000000.0,0.10,1.0,-8.072284e+08,0.0,inf


per-region MC draws: 200, ranking changed in 99%; MAPPO <= PriorityHeuristic on priority-weighted burden in 100% of draws


,kind,priority_delta,rank_changed,MAPPO_prio_mwh_day,MAPPO_ens_gwh,Priority_prio_mwh_day,Priority_ens_gwh,Proportional_prio_mwh_day,Proportional_ens_gwh,MAPPO_beats_priority,draw,prio_Dhaka,prio_Chattogram,prio_Khulna,prio_Rajshahi,prio_Sylhet,prio_Barishal,prio_Rangpur,prio_Mymensingh,prio_Cumilla
0,uniform,-0.2,False,1159.730,658.212,1351.228,976.007,1124.613,627.525,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,uniform,-0.1,False,1304.538,658.144,1520.132,976.007,1265.190,627.525,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,uniform,0.0,False,1449.332,658.077,1689.035,976.007,1405.767,627.525,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,uniform,0.1,False,1568.312,658.004,1840.731,976.007,1518.388,627.525,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,uniform,0.2,False,1644.221,657.922,1965.163,976.007,1586.844,627.525,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


reward sweep: 11 trainings in 658s


,d_alpha,d_beta,d_gamma,d_delta,d_epsilon,ens_gwh,priority_weighted_mwh_day,jain,over_dispatch_pct
0,0.0,0.0,0.0,0.0,0.0,658.5402,1457.8369,0.9998,2.5883
1,-0.2,0.0,0.0,0.0,0.0,658.5700,1457.7940,0.9998,2.5839
2,0.2,0.0,0.0,0.0,0.0,658.4777,1457.9552,0.9998,2.5896
3,0.0,-0.2,0.0,0.0,0.0,658.3787,1458.5006,0.9998,2.5865
4,0.0,0.2,0.0,0.0,0.0,658.6643,1457.3181,0.9998,2.5897
5,0.0,0.0,-0.2,0.0,0.0,658.5389,1457.8210,0.9998,2.5883
6,0.0,0.0,0.2,0.0,0.0,658.5411,1457.8538,0.9998,2.5883
7,0.0,0.0,0.0,-0.2,0.0,658.5208,1457.6539,0.9998,2.5897
8,0.0,0.0,0.0,0.2,0.0,658.5601,1458.0469,0.9998,2.5842
9,0.0,0.0,0.0,0.0,-0.2,658.5404,1457.8524,0.9998,2.5883


### 16.7  2025-2040 scenario projection

In [47]:
# Base year = 2024, the last full year of real data (ends 2024-12-30).
# Regional values are evening-peak MW (sub-station end).
real_only = realized[~realized["is_imputed"].astype(bool)]
base_year = (real_only[real_only["timestamp"].dt.year == 2024]
             .groupby("region")["demand_mw"].mean().reindex(REGIONS).to_dict())
print("2024 mean daily peak demand per region (MW):", {k: round(v) for k, v in base_year.items()})

assumptions = ScenarioAssumptions(supply_cagr=CFG.grid.supply_cagr,
                                  lstm_mape=fc_metrics["MAPE"] / 100.0)
clim = calibrate_from_history(realized, raw, assumptions)
print(f"calibrated weekend factor {assumptions.weekend_discount:.3f}, "
      f"capacity base {assumptions.supply_base_mw:,.0f} MW, "
      f"band from test MAPE {assumptions.lstm_mape:.2%}")
print("calibrated monthly factors (Dhaka):",
      {m: round(v, 3) for m, v in assumptions.monthly_factors["Dhaka"].items()})

P0, P1 = CFG.data.projection_start, CFG.data.projection_end
scenarios = project_all_scenarios(base_year, assumptions, P0, P1)
scenarios.to_csv(f"{OUTPUT_DIR}/scenario_projection_2025_2040.csv.gz", index=False,
                 compression="gzip")
proj_daily = {s: project_daily(base_year, assumptions, s, P0, P1, seed=i)
              for i, s in enumerate(("low", "central", "high"))}
daily_rows = []
for s, p in proj_daily.items():
    for i, r in enumerate(REGIONS):
        daily_rows.append(pd.DataFrame({"date": p["dates"].date, "region": r, "scenario": s,
                                        "peak_mw_mean": p["mean"][:, i],
                                        "peak_mw_lo90": p["lo90"][:, i],
                                        "peak_mw_hi90": p["hi90"][:, i]}))
pd.concat(daily_rows).to_csv(f"{OUTPUT_DIR}/scenario_projection_daily_peak_2025_2040.csv",
                             index=False)
print("Hourly rows:", len(scenarios))
display(scenarios.head())

2024 mean daily peak demand per region (MW): {'Dhaka': 4509, 'Chattogram': 1209, 'Khulna': 1515, 'Rajshahi': 1300, 'Sylhet': 490, 'Barishal': 383, 'Rangpur': 839, 'Mymensingh': 1058, 'Cumilla': 1163}
calibrated weekend factor 0.971, capacity base 14,876 MW, band from test MAPE 7.91%
calibrated monthly factors (Dhaka): {1: 0.794, 2: 0.834, 3: 0.966, 4: 1.052, 5: 1.075, 6: 1.085, 7: 1.076, 8: 1.089, 9: 1.127, 10: 1.076, 11: 0.975, 12: 0.852}
Hourly rows: 3786912


,date,region,hour,demand_mean,demand_lo90,demand_hi90,scenario
0,2025-01-01,Dhaka,0,2329.519043,2057.718750,2619.826660,low
1,2025-01-01,Dhaka,1,2329.519043,2057.718750,2619.826660,low
2,2025-01-01,Dhaka,2,2329.519043,2057.718750,2619.826660,low
3,2025-01-01,Dhaka,3,2329.519043,2057.718750,2619.826660,low
4,2025-01-01,Dhaka,4,2411.416016,2130.060547,2711.929932,low


In [48]:
import os
import zipfile

# Define the output folder and zip file name
output_folder = '/kaggle/working'  # Change this if your files are elsewhere
zip_filename = '/kaggle/working/output.zip'

# Create the zip file
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_folder):
        for file in files:
            file_path = os.path.join(root, file)
            # Skip the zip file itself to avoid recursion
            if file_path == zip_filename:
                continue
            # Store files with relative paths
            arcname = os.path.relpath(file_path, output_folder)
            zipf.write(file_path, arcname)
            print(f"Added: {arcname}")

print(f"\n✅ Zip created: {zip_filename}")
print(f"📦 Size: {os.path.getsize(zip_filename) / (1024*1024):.2f} MB")

Added: .virtual_documents/__notebook_source__.ipynb
Added: outputs/mappo_training_log.csv
Added: outputs/scheduling_daily_test.csv
Added: outputs/optuna_forecaster_best.json
Added: outputs/optuna_marl_best.json
Added: outputs/forecasting_baselines_comparison.csv
Added: outputs/scenario_projection_2025_2040.csv.gz
Added: outputs/scenario_projection_daily_peak_2025_2040.csv
Added: outputs/scheduling_comparison_test.csv
Added: outputs/forecaster_training_history.csv
Added: outputs/gnn_lstm_forecaster.pt
Added: outputs/sensitivity_economic.csv
Added: outputs/ablation_results_per_seed.csv
Added: outputs/optuna_marl_trials.csv
Added: outputs/ablation_results.csv
Added: outputs/optuna_forecaster_trials.csv
Added: outputs/forecasting_metrics_per_horizon.csv
Added: outputs/mappo_policy.pt
Added: outputs/scheduling_comparison_val.csv
Added: outputs/sensitivity_reward_weights.csv
Added: outputs/sensitivity_priority.csv
Added: outputs/forecasting_metrics.csv
Added: outputs/scheduling_paired_tests_

### 16.8  Daily per-region schedule (trained MAPPO policy) + Markdown report

proj_dates = proj_daily["central"]["dates"]
t0 = time.time()
# Embeddings the agents would observe on each projected day (computed once --
# the growth-deflated input path is identical across scenarios).
proj_emb = build_projection_embeddings(model, raw, proj_daily["central"]["seasonal"],
                                       proj_dates, clim, scaler, adj, edge_attr)
print(f"projection embeddings {proj_emb.shape} in {time.time()-t0:.0f}s")


def make_mappo_policy_fn(agent, emb, prio=PRIO_VEC):
    def policy_fn(demand, cap, dates):
        fc_peak = demand.max(-1)                                       # (D, N)
        trailing = (pd.DataFrame(fc_peak).shift(1).rolling(7, min_periods=1).mean()
                      .bfill().values)
        obs = build_obs_array(emb, fc_peak, cap, trailing, prio)
        return mappo_actions_batch(agent, obs) * demand
    return policy_fn


def supply_fn(dates):
    return project_supply(dates, assumptions)


def make_baseline_policy_fn(name):
    def policy_fn(demand, cap, dates):
        out = np.empty_like(demand)
        for d in range(len(demand)):
            if name == "ProportionalToForecast":
                out[d] = proportional_to_forecast(demand[d], cap[d])
            elif name == "PriorityHeuristic":
                out[d] = priority_heuristic(demand[d], cap[d], PRIO_VEC)
            elif name == "GreedySmallestFirst":
                out[d] = greedy_smallest_first(demand[d], cap[d])
            elif name == "HistoricalShare":
                out[d] = historical_share(demand[d], cap[d], HIST_SHARE)
        return out
    return policy_fn


# REPORT_POLICY: "auto" = lowest VALIDATION (2023) priority-weighted ENS among
# MAPPO and the implementable baselines (HistoricalRealized needs realised
# supply and the LP needs perfect foresight, so neither can schedule the
# future). Or force one, e.g. "MAPPO (GNN-LSTM obs)".
REPORT_POLICY = "auto"
candidates = ["MAPPO (GNN-LSTM obs)", "ProportionalToForecast", "PriorityHeuristic",
              "GreedySmallestFirst", "HistoricalShare"]
val_rank = (sched_val_summary.set_index("policy").loc[candidates, "priority_weighted_mwh_mean"]
            .sort_values())
chosen = val_rank.index[0] if REPORT_POLICY == "auto" else REPORT_POLICY
print("vzazaalidation priority-weighted ENS (MWh/day):", val_rank.round(1).to_dict())
print("report policy:", chosen)
OBS_EMBED_PROJ.update(MAIN_PROJ)
policy_fn = (make_mappo_policy_fn(mappo, proj_emb) if chosen.startswith("MAPPO")
             else make_baseline_policy_fn(chosen))
reports = {}
for s in ("central", "low", "high"):
    t0 = time.time()
    sub = scenarios[scenarios["scenario"] == s]
    reports[s] = generate_daily_report(sub, policy_fn, supply_fn, output_dir=OUTPUT_DIR,
                                       scenario=s, write_hourly=True)
    print(f"{s:8s} schedule: {len(reports[s]):,} region-days in {time.time()-t0:.0f}s")

write_markdown_summary(reports, OUTPUT_DIR, assumptions,
                       f"{chosen} (selected on 2023 validation, priority-weighted ENS)",
                       test_table=sched_summary)
schedule_df = reports["central"]
print("\nReport written. Files:")
for p in sorted(Path(OUTPUT_DIR).rglob("*")):
    if p.is_file():
        print(zf"  {p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size/1e6:.2f} MB)")
display(schedule_df[schedule_df["shed_mwh"] > 0].head(10))
print(Path(f"{OUTPUT_DIR}/daily_schedule_summary.md").read_text()[:3000])

## Done

All artifacts are saved under `/kaggle/working/outputs/` (`OUTPUT_DIR`, printed in Section 1) and are kept when you **Save Version**:

* `optuna_forecaster_best.json`, `optuna_forecaster_trials.csv`, `optuna_marl_best.json`, `optuna_marl_trials.csv`
* `gnn_lstm_forecaster.pt`, `mappo_policy.pt`, `forecaster_training_history.csv`, `mappo_training_log.csv`
* `forecasting_metrics.csv` (incl. MC-Dropout and conformal PICP/MPIW/Winkler), `forecasting_baselines_comparison.csv`, `forecasting_metrics_per_region.csv`, `forecasting_metrics_per_horizon.csv`
* `scheduling_comparison_test.csv`, `scheduling_paired_tests_test.csv`, `scheduling_daily_test.csv`
* `ablation_results.csv`, `ablation_results_per_seed.csv`
* `sensitivity_economic.csv`, `sensitivity_priority.csv`, `sensitivity_reward_weights.csv`
* `scenario_projection_2025_2040.csv.gz` (hourly), `scenario_projection_daily_peak_2025_2040.csv`
* `daily_schedule_2025_2040.csv` (central) + `_low` / `_high`, `daily_hourly_schedule_2025_2040_<scenario>.csv.gz`
* `daily_schedule_summary.md`
* `per_region/<Region>_yearly.csv` (central) + `_low` / `_high`